# 04 - Time-Safe Feature Engineering  
  
*AFL Matchday Demand Forecasting - Building Model-Ready Historical and Scoring Datasets*

## 1. Objective and Prediction Design

### 1.1 Notebook Objective

This notebook transforms the validated PostgreSQL staging tables into time-safe datasets for pre-match AFL attendance forecasting.

It combines historical match records with team, venue, and school-holiday information, then creates match-context and historical features that would have been available before each match began. Historical attendance and score information is shifted so that the current match cannot contribute to its own predictor values.

The notebook produces two datasets:

- a historical dataset containing pre-match attributes and the observed attendance target;
- a 2026 scoring dataset containing the same attributes for fixtures available in the 17 August 2026 Squiggle snapshot.

This notebook does not fit missing-value imputers, categorical encoders, feature scalers, or machine-learning models. Those steps will be completed in the model-development notebook using the training data only.

### 1.2 Prediction Target and Forecast Point

The prediction target is `attendance`, defined as the total number of spectators recorded for one completed AFL match. The modelling task is therefore a supervised regression problem in which each prediction represents the expected attendance for one scheduled fixture.

For historical matches, the forecast point is simulated by allowing each row to use only information that was available before that match began. The observed attendance remains the target, while current-match attendance and final scores are excluded from the predictor set.

For the 2026 scoring dataset, the forecast point is the Squiggle snapshot date of 17 August 2026. Only fixtures that had an assigned date, teams, and venue at that snapshot are eligible for scoring.

The available fixtures were scheduled between 20 and 23 August 2026, giving forecast lead times of approximately three to six days. Therefore, the current v1 workflow is described as snapshot-based pre-match forecasting rather than a fixed seven-day forecast.

Operational demand categories, such as low, medium, and high demand, may later be derived from the numerical attendance predictions. They are not the prediction target used in this notebook.

### 1.3 Feature Availability and Leakage Rules

An attribute is eligible only when its value would have been available at the relevant pre-match forecast point. This rule is applied separately to historical matches and the 2026 scoring fixtures.

#### 1.3.1 Directly Available Pre-Match Information

The following information is known before a scheduled match and may be used directly:

- season and reviewed round information;
- match date and scheduled start time;
- home team and away team;
- assigned venue and venue location;
- published school-holiday status;
- whether the fixture is a known finals match.

Calendar features such as month, day of the week, weekend status, and start hour may be derived from these scheduled values.

#### 1.3.2 Historical Information Requiring a Time Shift

Historical attendance and match results may influence future attendance, but they must be calculated from earlier matches only.

Eligible historical features include:

- prior home-team attendance;
- prior away-team attendance;
- prior venue attendance;
- previous team wins and losses;
- previous score margins;
- rolling attendance averages;
- rolling win rates and score-margin averages.

For every historical match, these features must be shifted before rolling calculations are applied. The current match must never contribute to its own historical predictors.

#### 1.3.3 Current-Match Outcomes Excluded from attributes

The following fields describe outcomes that are unknown before the match and must not be used as current-match predictors:

- current-match `attendance`;
- current-match `home_score`;
- current-match `away_score`;
- the current-match winner;
- current-match completion percentage or completion status.

The historical attendance remains the prediction target. Earlier scores and attendance values may be used only after they have been shifted into historical features.

#### 1.3.4 Identifier and Processing Fields

Fields such as `match_id`, `source_game_id`, and `snapshot_date` are retained for traceability but are not model attributes.

Team IDs are category identifiers rather than numerical measurements. The modelling pipeline will use the corresponding team fields as categorical features rather than treating the ID values as continuous numbers.

#### 1.3.5 Features Outside the Current v1 Scope

The following potentially useful predictors are not included because reliable pre-match values are not currently available in the selected data sources:

- expected player line-ups;
- player injuries, suspensions, and late withdrawals;
- player-level recent performance;
- ticket prices and ticket-sales progress;
- membership and marketing activity;
- actual matchday weather.

These variables are documented as possible future improvements rather than being approximated with unreliable values.

#### 1.3.6 Scoring-Date Rule

Historical features for a 2026 fixture must use only records available by the 17 August 2026 snapshot date and before the scheduled fixture.

Because Squiggle does not provide attendance, 2026 team-form features may use completed 2026 match scores, while attendance-history features use the latest available historical attendance records through 2025.

### 1.4 Expected Outputs and Scope Boundaries

This notebook produces two model-oriented datasets from the validated PostgreSQL staging tables.

| Output dataset | Data grain | Main purpose |
|---|---|---|
| `historical_model_dataset.csv` | One eligible completed historical match per row | Provide pre-match attributes and the observed attendance target for model development |
| `squiggle_2026_scoring_dataset.csv` | One scoreable fixture from the 17 August 2026 snapshot per row | Provide the same attribute structure for fixtures without an observed attendance target |

The files will be exported to:

```text
data/processed/historical_model_dataset.csv
data/processed/squiggle_2026_scoring_dataset.csv

The historical dataset will contain:

- match identifiers and time-period labels;
- the observed `attendance` target;
- scheduled match, team, and venue information;
- calendar and school-holiday features;
- time-safe attendance-history features;
- time-safe team-form features.

The scoring dataset will contain:

- snapshot and fixture identifiers;
- the same scheduled match, calendar, team, and venue features;
- the latest historical features available at the snapshot date;
- no observed `attendance` target.

The following processing boundaries apply:

- the five PostgreSQL staging tables are queried but not modified;
- no additional PostgreSQL tables are created in this notebook;
- files under `data/raw/` and `data/interim/postgres_ready/` remain unchanged;
- current-match attendance and final scores are excluded from the predictor set;
- model-specific missing-value imputation, categorical encoding, and scaling are deferred to the model-development pipeline;
- machine-learning training, model comparison, prediction, and demand-band assignment are outside this notebook;
- player line-ups, player-level performance, injuries, ticketing, marketing, and actual matchday weather remain outside the current v1 feature scope.

The next notebook will use these two exported datasets to fit preprocessing pipelines, compare regression models, and evaluate attendance forecasting performance.

## 2. Load the Modeling Sources from PostgreSQL

### 2.1 Processing Environment and Database Connection

This section imports the packages required for feature engineering, locates the project root, and loads the PostgreSQL connection settings from the local `.env` file.

A short read-only query confirms that the notebook can connect to the expected database. Database credentials are not displayed, and the test connection is closed immediately after use.

In [1]:
from pathlib import Path
import os

import pandas as pd
import psycopg
from dotenv import load_dotenv


# Resolve the project root whether the notebook starts from the
# repository root or from the notebooks directory.
project_root = Path.cwd().resolve()

if project_root.name.lower() == "notebooks":
    project_root = project_root.parent

env_path = project_root / ".env"

if not env_path.exists():
    raise FileNotFoundError(f"Environment file not found: {env_path}")

# Load the local database settings without displaying credentials.
load_dotenv(env_path)

required_settings = {
    "DB_HOST": os.getenv("DB_HOST"),
    "DB_PORT": os.getenv("DB_PORT"),
    "DB_NAME": os.getenv("DB_NAME"),
    "DB_USER": os.getenv("DB_USER"),
    "DB_PASSWORD": os.getenv("DB_PASSWORD"),
}

missing_settings = [
    setting_name
    for setting_name, setting_value in required_settings.items()
    if setting_value is None or setting_value == ""
]

if missing_settings:
    raise ValueError(
        "Missing required database settings: "
        + ", ".join(missing_settings)
    )

# Use the same connection configuration throughout this notebook.
db_config = {
    "host": required_settings["DB_HOST"],
    "port": required_settings["DB_PORT"],
    "dbname": required_settings["DB_NAME"],
    "user": required_settings["DB_USER"],
    "password": required_settings["DB_PASSWORD"],
}

# Test the connection with a read-only database identity query.
with psycopg.connect(**db_config) as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT
                current_database(),
                current_user;
            """
        )
        connected_database, connected_user = cursor.fetchone()

print(f"Project root: {project_root}")
print(f"Database: {connected_database}")
print(f"Database user: {connected_user}")
print("PostgreSQL connection check: PASSED")

Project root: D:\Projects\AFL_Project
Database: afl_project
Database user: afl_app
PostgreSQL connection check: PASSED


### 2.2 Historical Match Data

The historical match table contains one completed AFL match per row and provides the observed attendance target required for model development.

All validated staging columns are loaded at this point. The attribute columns will be selected later, after the match-context and historical features have been created.

The loading check confirms that:

- the table is not empty;
- the required modelling fields are present;
- `match_id` remains unique;
- the attendance target is available;
- the match-date coverage can be identified.

The row count is reported rather than fixed because additional historical matches may be included in a future data refresh.

In [2]:
def read_postgres_query(query):
    """Execute a read-only PostgreSQL query and return a pandas DataFrame."""
    with psycopg.connect(**db_config) as connection:
        with connection.cursor() as cursor:
            cursor.execute(query)
            records = cursor.fetchall()
            column_names = [
                column.name
                for column in cursor.description
            ]

    return pd.DataFrame(records, columns=column_names)


# Load all validated fields from the historical match staging table.
historical_matches = read_postgres_query(
    """
    SELECT *
    FROM staging.historical_matches
    ORDER BY match_date, match_id;
    """
)

if historical_matches.empty:
    raise ValueError("The historical match table contains no records.")

required_historical_columns = {
    "match_id",
    "season_year",
    "round_label",
    "match_date",
    "start_time",
    "home_team_id",
    "home_team",
    "away_team_id",
    "away_team",
    "venue_name",
    "attendance",
    "home_score",
    "away_score",
}

missing_historical_columns = sorted(
    required_historical_columns
    - set(historical_matches.columns)
)

if missing_historical_columns:
    raise ValueError(
        "Missing required historical columns: "
        + ", ".join(missing_historical_columns)
    )

# Convert the SQL date values to a consistent pandas datetime type.
historical_matches["match_date"] = pd.to_datetime(
    historical_matches["match_date"],
    errors="raise",
)

duplicate_match_ids = int(
    historical_matches["match_id"].duplicated().sum()
)

missing_attendance_targets = int(
    historical_matches["attendance"].isna().sum()
)

if duplicate_match_ids > 0:
    raise ValueError(
        f"Duplicate historical match identifiers found: {duplicate_match_ids}"
    )

if missing_attendance_targets > 0:
    raise ValueError(
        f"Missing historical attendance targets found: "
        f"{missing_attendance_targets}"
    )

print("Source table: staging.historical_matches")
print(f"Records loaded: {len(historical_matches):,}")
print(f"Columns loaded: {historical_matches.shape[1]}")
print(
    "Match-date range: "
    f"{historical_matches['match_date'].min().date()} to "
    f"{historical_matches['match_date'].max().date()}"
)
print(f"Duplicate match_id rows: {duplicate_match_ids}")
print(f"Missing attendance targets: {missing_attendance_targets}")

display(
    historical_matches[
        [
            "match_id",
            "season_year",
            "round_label",
            "match_date",
            "home_team",
            "away_team",
            "venue_name",
            "attendance",
        ]
    ].head(3)
)

print("Historical match loading check: PASSED")

Source table: staging.historical_matches
Records loaded: 2,879
Columns loaded: 19
Match-date range: 2012-03-24 to 2025-09-27
Duplicate match_id rows: 0
Missing attendance targets: 0


,match_id,season_year,round_label,match_date,home_team,away_team,venue_name,attendance
0,20120324_H09_A16,2012,Round 1,2012-03-24,Greater Western Sydney,Sydney,Stadium Australia,38203
1,20120329_H14_A03,2012,Round 1,2012-03-29,Richmond,Carlton,MCG,78285
2,20120330_H10_A04,2012,Round 1,2012-03-30,Hawthorn,Collingwood,MCG,78466


Historical match loading check: PASSED


### 2.3 2026 Match Snapshot and Supporting Reference Data

The 2026 Squiggle snapshot provides the completed matches, scheduled fixtures, and finals placeholder records available on 17 August 2026.

Three supporting reference tables are also loaded:

- the team reference table provides consistent AFL team identifiers and names;
- the venue reference table connects each venue to its location and school-holiday city;
- the daily school-holiday table provides the holiday status for each capital city and calendar date.

These tables were validated during the PostgreSQL loading workflow. This section therefore performs only a concise loading check rather than repeating the complete database validation.

In [3]:
# Load the 2026 Squiggle match snapshot.
squiggle_matches_2026 = read_postgres_query(
    """
    SELECT *
    FROM staging.squiggle_matches_2026_snapshot
    ORDER BY snapshot_date, source_game_id;
    """
)

# Load the supporting team, venue, and school-holiday tables.
team_reference = read_postgres_query(
    """
    SELECT *
    FROM staging.team_reference
    ORDER BY team_id;
    """
)

venue_reference = read_postgres_query(
    """
    SELECT *
    FROM staging.venue_reference
    ORDER BY venue_name;
    """
)

school_holidays_daily = read_postgres_query(
    """
    SELECT *
    FROM staging.school_holidays_daily
    ORDER BY city, calendar_date;
    """
)

loaded_sources = {
    "staging.squiggle_matches_2026_snapshot": squiggle_matches_2026,
    "staging.team_reference": team_reference,
    "staging.venue_reference": venue_reference,
    "staging.school_holidays_daily": school_holidays_daily,
}

# Confirm that every required source returned at least one record.
empty_sources = [
    table_name
    for table_name, data_frame in loaded_sources.items()
    if data_frame.empty
]

if empty_sources:
    raise ValueError(
        "The following PostgreSQL tables contain no records: "
        + ", ".join(empty_sources)
    )

# Convert SQL date values to consistent pandas datetime types.
squiggle_matches_2026["snapshot_date"] = pd.to_datetime(
    squiggle_matches_2026["snapshot_date"],
    errors="raise",
)

squiggle_matches_2026["match_date"] = pd.to_datetime(
    squiggle_matches_2026["match_date"],
    errors="coerce",
)

school_holidays_daily["calendar_date"] = pd.to_datetime(
    school_holidays_daily["calendar_date"],
    errors="raise",
)

# Summarise the loaded modeling sources without fixing expected row counts.
display(
    pd.DataFrame(
        [
            {
                "source_table": table_name,
                "records": len(data_frame),
                "columns": data_frame.shape[1],
            }
            for table_name, data_frame in loaded_sources.items()
        ]
    )
)

print("Squiggle snapshot record types:")

display(
    squiggle_matches_2026["snapshot_record_type"]
    .value_counts(dropna=False)
    .rename_axis("snapshot_record_type")
    .reset_index(name="records")
)

print("Supporting modeling-source loading check: PASSED")

,source_table,records,columns
0,staging.squiggle_matches_2026_snapshot,218,37
1,staging.team_reference,18,3
2,staging.venue_reference,27,5
3,staging.school_holidays_daily,67208,5


Squiggle snapshot record types:


,snapshot_record_type,records
0,completed_match,198
1,finals_placeholder,11
2,future_fixture,9


Supporting modeling-source loading check: PASSED


## 3. Build the Base Historical Match Dataset

### 3.1 Select Eligibile Completed Matches

The historical feature base retains completed matches with the information required for match-context features, attendance modeling, and historical team-form calculations.

A retained match must contain:

- a valid match identifier and match date;
- a scheduled start time;
- assigned home and away teams;
- an assigned venue;
- a recorded attendance value;
- final home and away scores.

Zero attendance is retained as an observed value rather than being treated as missing data. In this dataset, the zero-attendance matches occur during the pandemic-affected 2020 and 2021 seasons.

The pandemic affected both spectator attendance and match conditions. However, these two types of information serve different purposes in the forecasting workflow:

- attendance from 2020 and 2021 will not be used as a model target or as a source for normal attendance-history features because venue closures and capacity restrictions directly limited the recorded crowds;
- match results from the same period are retained as publicly observed results that were available before later fixtures;
- retaining these results does not assume that team performance was unaffected by the pandemic;
- their possible effect on team-form measures is documented as a project limitation.

No season is physically removed from the working history at this stage. The 2020 and 2021 rows remain available for chronological team-form calculations, while their final dataset role will be assigned later.

The temporary final-score columns are required to create shifted team-form features. Current-match scores will not be included in the final predictor set.

In [4]:
# Retain the fields required for match context, the attendance target,
# and later historical feature calculations.
historical_base_columns = [
    "match_id",
    "source_game_id",
    "season_year",
    "round_label",
    "match_date",
    "start_time",
    "home_team_id",
    "home_team",
    "away_team_id",
    "away_team",
    "venue_name",
    "attendance",
    "home_score",
    "away_score",
]

historical_match_base = historical_matches[
    historical_base_columns
].copy()

# A completed match requires valid identity, schedule, team, venue,
# attendance, and final-score information. Zero attendance remains valid.
critical_completed_match_fields = [
    "match_id",
    "match_date",
    "start_time",
    "home_team_id",
    "home_team",
    "away_team_id",
    "away_team",
    "venue_name",
    "attendance",
    "home_score",
    "away_score",
]

has_complete_match_fields = (
    historical_match_base[critical_completed_match_fields]
    .notna()
    .all(axis=1)
)

has_different_teams = (
    historical_match_base["home_team_id"]
    .ne(historical_match_base["away_team_id"])
)

valid_completed_match_mask = (
    has_complete_match_fields
    & has_different_teams
)

invalid_historical_matches = historical_match_base.loc[
    ~valid_completed_match_mask
].copy()

# Stop rather than silently removing an unexpected invalid match.
if not invalid_historical_matches.empty:
    display(
        invalid_historical_matches[
            [
                "match_id",
                "match_date",
                "home_team",
                "away_team",
                "venue_name",
                "attendance",
                "home_score",
                "away_score",
            ]
        ].head(10)
    )

    raise ValueError(
        f"Invalid completed historical matches found: "
        f"{len(invalid_historical_matches):,}"
    )

historical_match_base = (
    historical_match_base.loc[valid_completed_match_mask]
    .sort_values(["match_date", "match_id"])
    .reset_index(drop=True)
)

# Add factual flags without changing the original attendance values.
historical_match_base["is_pandemic_affected_season"] = (
    historical_match_base["season_year"].isin([2020, 2021])
)

historical_match_base["has_positive_attendance"] = (
    historical_match_base["attendance"].gt(0)
)

zero_attendance_by_year = (
    historical_match_base.loc[
        ~historical_match_base["has_positive_attendance"]
    ]
    .groupby("season_year")
    .size()
    .rename("zero_attendance_matches")
    .reset_index()
)

selection_summary = pd.DataFrame(
    [
        {
            "record_group": "Historical records loaded",
            "records": len(historical_matches),
        },
        {
            "record_group": "Completed matches retained",
            "records": len(historical_match_base),
        },
        {
            "record_group": "Pandemic-affected matches retained",
            "records": int(
                historical_match_base[
                    "is_pandemic_affected_season"
                ].sum()
            ),
        },
        {
            "record_group": "Zero-attendance matches retained",
            "records": int(
                (~historical_match_base[
                    "has_positive_attendance"
                ]).sum()
            ),
        },
        {
            "record_group": "Invalid records excluded",
            "records": len(invalid_historical_matches),
        },
    ]
)

display(selection_summary)

print("Zero-attendance matches by season:")
display(zero_attendance_by_year)

display(
    historical_match_base[
        [
            "match_id",
            "season_year",
            "match_date",
            "home_team",
            "away_team",
            "attendance",
            "home_score",
            "away_score",
            "is_pandemic_affected_season",
            "has_positive_attendance",
        ]
    ].head(3)
)

print("Completed historical match selection check: PASSED")

,record_group,records
0,Historical records loaded,2879
1,Completed matches retained,2879
2,Pandemic-affected matches retained,369
3,Zero-attendance matches retained,66
4,Invalid records excluded,0


Zero-attendance matches by season:


,season_year,zero_attendance_matches
0,2020,29
1,2021,37


,match_id,season_year,match_date,home_team,away_team,attendance,home_score,away_score,is_pandemic_affected_season,has_positive_attendance
0,20120324_H09_A16,2012,2012-03-24,Greater Western Sydney,Sydney,38203,37,100,False,True
1,20120329_H14_A03,2012,2012-03-29,Richmond,Carlton,78285,81,125,False,True
2,20120330_H10_A04,2012,2012-03-30,Hawthorn,Collingwood,78466,137,115,False,True


Completed historical match selection check: PASSED


### 3.2 Add Venue Location and School-Holiday Information

Venue location fields are added by joining each historical match to the standardised venue reference table.

The assigned school-holiday city and match date are then used to retrieve the published daily school-holiday status.

The data contains six international matches played in Wellington and Shanghai. These records do not have an Australian school-holiday calendar, so their holiday values remain missing rather than being incorrectly classified as non-holiday dates.

International matches remain in the working history for later team-form calculations, but they will not be used as domestic attendance target rows or as sources for domestic attendance-history features.

Both joins use many-to-one validation to preserve the one-row-per-match grain.

In [5]:
# Add the standardised location and calendar assignment for each venue.
historical_match_context = historical_match_base.merge(
    venue_reference[
        [
            "venue_name",
            "venue_city",
            "state_code",
            "country_code",
            "school_holiday_city",
        ]
    ],
    on="venue_name",
    how="left",
    validate="many_to_one",
)

missing_venue_count = int(
    historical_match_context["venue_city"].isna().sum()
)

if missing_venue_count > 0:
    raise ValueError(
        f"Historical matches with unmapped venues: "
        f"{missing_venue_count}"
    )

# Identify whether the match was played in Australia.
historical_match_context["is_domestic_match"] = (
    historical_match_context["country_code"].eq("AU")
)

# Prepare one school-holiday record for each calendar city and date.
holiday_lookup = (
    school_holidays_daily[
        [
            "city",
            "calendar_date",
            "is_school_holiday",
        ]
    ]
    .rename(
        columns={
            "city": "school_holiday_city",
            "calendar_date": "match_date",
        }
    )
)

# Add the school-holiday status without assigning false values
# to international matches that do not use an Australian calendar.
historical_match_context = historical_match_context.merge(
    holiday_lookup,
    on=["school_holiday_city", "match_date"],
    how="left",
    validate="many_to_one",
)

missing_domestic_holiday_count = int(
    (
        historical_match_context["is_domestic_match"]
        & historical_match_context["is_school_holiday"].isna()
    ).sum()
)

if missing_domestic_holiday_count > 0:
    raise ValueError(
        "Australian matches with missing school-holiday records: "
        f"{missing_domestic_holiday_count}"
    )

# Use a nullable Boolean type because international matches
# intentionally have no Australian school-holiday value.
historical_match_context["is_school_holiday"] = (
    historical_match_context["is_school_holiday"]
    .astype("boolean")
)

domestic_match_count = int(
    historical_match_context["is_domestic_match"].sum()
)

international_match_count = int(
    (~historical_match_context["is_domestic_match"]).sum()
)

print(f"Historical match rows: {len(historical_match_context):,}")
print(f"Australian match rows: {domestic_match_count:,}")
print(f"International match rows: {international_match_count:,}")
print(f"Unmapped venues: {missing_venue_count}")
print(
    "Missing Australian holiday records: "
    f"{missing_domestic_holiday_count}"
)

display(
    historical_match_context[
        [
            "match_id",
            "match_date",
            "venue_name",
            "venue_city",
            "country_code",
            "school_holiday_city",
            "is_school_holiday",
            "is_domestic_match",
        ]
    ].head(3)
)

print("International venue summary:")

display(
    historical_match_context.loc[
        ~historical_match_context["is_domestic_match"],
        ["venue_name", "venue_city", "country_code"],
    ]
    .value_counts()
    .rename("matches")
    .reset_index()
)

print("Historical venue and school-holiday join check: PASSED")

Historical match rows: 2,879
Australian match rows: 2,873
International match rows: 6
Unmapped venues: 0
Missing Australian holiday records: 0


,match_id,match_date,venue_name,venue_city,country_code,school_holiday_city,is_school_holiday,is_domestic_match
0,20120324_H09_A16,2012-03-24,Stadium Australia,Sydney,AU,Sydney,False,True
1,20120329_H14_A03,2012-03-29,MCG,Melbourne,AU,Melbourne,False,True
2,20120330_H10_A04,2012-03-30,MCG,Melbourne,AU,Melbourne,False,True


International venue summary:


,venue_name,venue_city,country_code,matches
0,Wellington,Wellington,NZ,3
1,Jiangwan Stadium,Shanghai,CN,3


Historical venue and school-holiday join check: PASSED


### 3.3 Create Calendar and Match-Context Features

A small set of scheduled match features is derived from the match date, start time, and reviewed round label.

These values would have been known before each match and are therefore safe to use as predictors:

- `match_month` identifies the calendar month;
- `match_day_of_week` identifies the scheduled weekday;
- `is_weekend` identifies Saturday and Sunday fixtures;
- `start_hour` converts the scheduled time into a numerical hour;
- `is_night_match` identifies matches starting at or after 5:00 PM;
- `is_finals_match` identifies reviewed round labels containing the word “Final”.

These rules are intentionally simple and transparent so that the same transformations can later be applied to the 2026 scoring fixtures.

In [6]:
# Preserve the joined match data before adding model-oriented features.
historical_match_features = historical_match_context.copy()

# Derive calendar features from information known before the match.
historical_match_features["match_month"] = (
    historical_match_features["match_date"]
    .dt.month
    .astype("Int64")
)

historical_match_features["match_day_of_week"] = (
    historical_match_features["match_date"]
    .dt.day_name()
    .astype("string")
)

historical_match_features["is_weekend"] = (
    historical_match_features["match_date"]
    .dt.dayofweek
    .isin([5, 6])
)

# Convert the scheduled start time to a continuous numerical hour.
# For example, 19:30 becomes 19.5.
historical_match_features["start_hour"] = (
    historical_match_features["start_time"]
    .map(
        lambda match_time:
        match_time.hour + match_time.minute / 60
    )
)

historical_match_features["is_night_match"] = (
    historical_match_features["start_hour"].ge(17)
)

# Group the different finals round labels into one match-context flag.
historical_match_features["is_finals_match"] = (
    historical_match_features["round_label"]
    .astype("string")
    .str.contains(
        "Final",
        case=False,
        na=False,
    )
)

display(
    historical_match_features[
        [
            "match_id",
            "match_date",
            "start_time",
            "round_label",
            "match_month",
            "match_day_of_week",
            "is_weekend",
            "start_hour",
            "is_night_match",
            "is_finals_match",
            "is_school_holiday",
        ]
    ].head(5)
)

print(
    f"Weekend matches: "
    f"{historical_match_features['is_weekend'].sum():,}"
)
print(
    f"Night matches: "
    f"{historical_match_features['is_night_match'].sum():,}"
)
print(
    f"Finals matches: "
    f"{historical_match_features['is_finals_match'].sum():,}"
)
print("Calendar and match-context feature creation: PASSED")

,match_id,match_date,start_time,round_label,match_month,match_day_of_week,is_weekend,start_hour,is_night_match,is_finals_match,is_school_holiday
0,20120324_H09_A16,2012-03-24,19:20:00,Round 1,3,Saturday,True,19.333333,True,False,False
1,20120329_H14_A03,2012-03-29,19:45:00,Round 1,3,Thursday,False,19.750000,True,False,False
2,20120330_H10_A04,2012-03-30,19:50:00,Round 1,3,Friday,False,19.833333,True,False,False
3,20120331_H06_A07,2012-03-31,16:45:00,Round 1,3,Saturday,True,16.750000,False,False,False
4,20120331_H08_A01,2012-03-31,15:45:00,Round 1,3,Saturday,True,15.750000,False,False,False


Weekend matches: 2,287
Night matches: 1,168
Finals matches: 126
Calendar and match-context feature creation: PASSED


## 4. Create Time-Safe Historical Features

### 4.1 Order Matches Chronologically

Historical features must be calculated in chronological order so that each match uses only information from earlier matches.

For team-level calculations, the one-row-per-match table is temporarily reshaped into a team-match history containing two rows for every match:

- one row from the home-team perspective;
- one row from the away-team perspective.

Each temporary row contains the relevant team, opponent, team score, opponent score, attendance, and match context. This allows the same rolling logic to be applied consistently to both home and away teams.

The final model dataset will return to one row per match. The two-row structure is used only for historical feature calculation.

No rolling calculation is performed in this step. The required `shift` operations are applied in the following sections.

In [7]:
# Create an exact local match timestamp for chronological ordering.
historical_feature_working = historical_match_features.copy()

historical_feature_working["match_datetime_local"] = pd.to_datetime(
    historical_feature_working["match_date"].dt.strftime("%Y-%m-%d")
    + " "
    + historical_feature_working["start_time"].astype("string"),
    errors="raise",
)

historical_feature_working = (
    historical_feature_working
    .sort_values(["match_datetime_local", "match_id"])
    .reset_index(drop=True)
)

team_history_source_columns = [
    "match_id",
    "match_datetime_local",
    "match_date",
    "season_year",
    "venue_name",
    "attendance",
    "home_team_id",
    "home_team",
    "away_team_id",
    "away_team",
    "home_score",
    "away_score",
    "is_pandemic_affected_season",
    "has_positive_attendance",
    "is_domestic_match",
]

# Create one temporary history row from the home-team perspective.
home_team_history = (
    historical_feature_working[team_history_source_columns]
    .rename(
        columns={
            "home_team_id": "team_id",
            "home_team": "team_name",
            "away_team_id": "opponent_id",
            "away_team": "opponent_name",
            "home_score": "team_score",
            "away_score": "opponent_score",
        }
    )
    .assign(team_role="home")
)

# Create one temporary history row from the away-team perspective.
away_team_history = (
    historical_feature_working[team_history_source_columns]
    .rename(
        columns={
            "away_team_id": "team_id",
            "away_team": "team_name",
            "home_team_id": "opponent_id",
            "home_team": "opponent_name",
            "away_score": "team_score",
            "home_score": "opponent_score",
        }
    )
    .assign(team_role="away")
)

team_match_history = pd.concat(
    [home_team_history, away_team_history],
    ignore_index=True,
)

# Create result fields that will later support shifted team-form features.
team_match_history["score_margin"] = (
    team_match_history["team_score"]
    - team_match_history["opponent_score"]
)

team_match_history["team_won"] = (
    team_match_history["score_margin"].gt(0)
)

team_match_history = (
    team_match_history
    .sort_values(
        [
            "team_id",
            "match_datetime_local",
            "match_id",
        ]
    )
    .reset_index(drop=True)
)

expected_team_history_rows = (
    len(historical_feature_working) * 2
)

duplicate_team_match_rows = int(
    team_match_history.duplicated(
        subset=["match_id", "team_id"]
    ).sum()
)

if len(team_match_history) != expected_team_history_rows:
    raise ValueError(
        "The team-match reshape did not produce "
        "two rows for every historical match."
    )

if duplicate_team_match_rows > 0:
    raise ValueError(
        f"Duplicate team-match rows found: "
        f"{duplicate_team_match_rows}"
    )

print(
    f"Historical match rows: "
    f"{len(historical_feature_working):,}"
)
print(
    f"Temporary team-history rows: "
    f"{len(team_match_history):,}"
)
print(
    f"Duplicate team-match rows: "
    f"{duplicate_team_match_rows}"
)

display(
    team_match_history[
        [
            "match_id",
            "match_datetime_local",
            "team_id",
            "team_name",
            "team_role",
            "opponent_name",
            "team_score",
            "opponent_score",
            "score_margin",
            "team_won",
        ]
    ].head(6)
)

print("Chronological team-match history construction: PASSED")

Historical match rows: 2,879
Temporary team-history rows: 5,758
Duplicate team-match rows: 0


,match_id,match_datetime_local,team_id,team_name,team_role,opponent_name,team_score,opponent_score,score_margin,team_won
0,20120331_H08_A01,2012-03-31 15:45:00,1,Adelaide,away,Gold Coast,137,68,69,True
1,20120407_H01_A18,2012-04-07 19:10:00,1,Adelaide,home,Western Bulldogs,82,64,18,True
2,20120415_H10_A01,2012-04-15 15:15:00,1,Adelaide,away,Hawthorn,84,140,-56,False
3,20120421_H01_A09,2012-04-21 16:10:00,1,Adelaide,home,Greater Western Sydney,96,50,46,True
4,20120429_H01_A13,2012-04-29 16:10:00,1,Adelaide,home,Port Adelaide,110,91,19,True
5,20120505_H16_A01,2012-05-05 19:40:00,1,Adelaide,away,Sydney,99,94,5,True


Chronological team-match history construction: PASSED


### 4.2 Create Team Attendance History Features

Recent attendance at matches involving a team may provide information about that team’s current ability to attract spectators.

For each team, this section calculates the mean attendance from its previous five matches, including matches where the team played at home or away.

A five-match window is used as a simple v1 measure of recent spectator interest. It reflects recent attendance patterns without allowing one unusually high or low crowd to dominate the feature. A longer window could respond too slowly to recent changes, while a one-match feature would be highly sensitive to unusual match conditions.

The five-match window was selected before model evaluation. It was not chosen by comparing results on the locked test period.

The attendance-history calculation excludes:

- matches from the pandemic-affected 2020 and 2021 seasons;
- international matches outside the Australian forecasting scope;
- matches with zero recorded attendance.

These exclusions apply only to attendance-history calculations. The original attendance values remain unchanged in the working data.

A one-match shift is applied before calculating the rolling mean. This ensures that the current match attendance cannot contribute to its own predictor value.

When fewer than five valid earlier observations are available, the feature uses the available earlier observations. If no valid earlier attendance exists, the feature remains missing and will be handled later by the model preprocessing pipeline.

In [8]:
# Identify attendance observations that represent normal domestic demand.
team_match_history["attendance_history_eligible"] = (
    team_match_history["is_domestic_match"]
    & ~team_match_history["is_pandemic_affected_season"]
    & team_match_history["has_positive_attendance"]
)

# Preserve the original attendance while masking observations that
# should not contribute to normal attendance-history features.
team_match_history["attendance_for_history"] = (
    team_match_history["attendance"].where(
        team_match_history["attendance_history_eligible"]
    )
)

# Shift before rolling so that the current match cannot use
# its own observed attendance.
team_match_history["team_last_5_attendance_mean"] = (
    team_match_history
    .groupby("team_id", sort=False)["attendance_for_history"]
    .transform(
        lambda attendance:
        attendance
        .shift(1)
        .rolling(
            window=5,
            min_periods=1,
        )
        .mean()
    )
)

# Return the temporary team-level feature to one row per match,
# with separate values for the home and away teams.
team_attendance_features = (
    team_match_history[
        [
            "match_id",
            "team_role",
            "team_last_5_attendance_mean",
        ]
    ]
    .pivot(
        index="match_id",
        columns="team_role",
        values="team_last_5_attendance_mean",
    )
    .rename(
        columns={
            "home": "home_team_last_5_attendance_mean",
            "away": "away_team_last_5_attendance_mean",
        }
    )
    .reset_index()
)

team_attendance_features.columns.name = None

historical_feature_working = historical_feature_working.merge(
    team_attendance_features,
    on="match_id",
    how="left",
    validate="one_to_one",
)

eligible_attendance_history_rows = int(
    team_match_history["attendance_history_eligible"].sum()
)

excluded_attendance_history_rows = int(
    (~team_match_history["attendance_history_eligible"]).sum()
)

print(
    "Eligible team-attendance history rows: "
    f"{eligible_attendance_history_rows:,}"
)
print(
    "Team-attendance history rows excluded from rolling calculations: "
    f"{excluded_attendance_history_rows:,}"
)

display(
    team_match_history[
        [
            "match_id",
            "match_date",
            "team_name",
            "team_role",
            "attendance",
            "attendance_for_history",
            "team_last_5_attendance_mean",
        ]
    ].head(8)
)

display(
    historical_feature_working[
        [
            "match_id",
            "match_date",
            "home_team",
            "away_team",
            "attendance",
            "home_team_last_5_attendance_mean",
            "away_team_last_5_attendance_mean",
        ]
    ].head(5)
)

print("Time-safe team attendance feature creation: PASSED")

Eligible team-attendance history rows: 5,008
Team-attendance history rows excluded from rolling calculations: 750


,match_id,match_date,team_name,team_role,attendance,attendance_for_history,team_last_5_attendance_mean
0,20120331_H08_A01,2012-03-31,Adelaide,away,12790,12790.0,NaN
1,20120407_H01_A18,2012-04-07,Adelaide,home,34021,34021.0,12790.000000
2,20120415_H10_A01,2012-04-15,Adelaide,away,33524,33524.0,23405.500000
3,20120421_H01_A09,2012-04-21,Adelaide,home,28261,28261.0,26778.333333
4,20120429_H01_A13,2012-04-29,Adelaide,home,41649,41649.0,27149.000000
5,20120505_H16_A01,2012-05-05,Adelaide,away,23969,23969.0,30049.000000
6,20120512_H01_A07,2012-05-12,Adelaide,home,35535,35535.0,32284.800000
7,20120520_H03_A01,2012-05-20,Adelaide,away,35917,35917.0,32587.600000


,match_id,match_date,home_team,away_team,attendance,home_team_last_5_attendance_mean,away_team_last_5_attendance_mean
0,20120324_H09_A16,2012-03-24,Greater Western Sydney,Sydney,38203,NaN,NaN
1,20120329_H14_A03,2012-03-29,Richmond,Carlton,78285,NaN,NaN
2,20120330_H10_A04,2012-03-30,Hawthorn,Collingwood,78466,NaN,NaN
3,20120331_H11_A02,2012-03-31,Melbourne,Brisbane Lions,33473,NaN,NaN
4,20120331_H08_A01,2012-03-31,Gold Coast,Adelaide,12790,NaN,NaN


Time-safe team attendance feature creation: PASSED


### 4.3 Create Venue Attendance History Features

Recent attendance at the same venue provides a simple measure of the venue’s typical crowd level. This may reflect venue capacity, location, accessibility, and the usual scale of matches hosted there.

For each historical match, this section calculates the mean attendance from the previous ten matches played at the same venue.

A ten-match window is used because venue attendance patterns generally change more slowly than short-term team interest. The longer window provides a more stable venue baseline and reduces the effect of one unusual match or opponent.

The ten-match window was selected before model evaluation. It was not chosen by comparing results on the locked test period.

The venue-attendance calculation excludes:

- matches from the pandemic-affected 2020 and 2021 seasons;
- international matches outside the Australian forecasting scope;
- matches with zero recorded attendance.

These exclusions apply only to the venue-attendance history. The original attendance values remain unchanged in the working data.

A one-match shift is applied before calculating the rolling mean. This ensures that the current match attendance cannot contribute to its own venue-history feature.

When fewer than ten valid earlier observations are available, the feature uses the available earlier observations. If the venue has no valid earlier attendance, the feature remains missing. These rows are retained, and the missing numerical values will later be handled by a preprocessing pipeline fitted on the training data only.

In [9]:
# Create a chronologically ordered venue-match history.
venue_match_history = (
    historical_feature_working[
        [
            "match_id",
            "match_datetime_local",
            "match_date",
            "venue_name",
            "attendance",
            "is_domestic_match",
            "is_pandemic_affected_season",
            "has_positive_attendance",
        ]
    ]
    .sort_values(
        [
            "venue_name",
            "match_datetime_local",
            "match_id",
        ]
    )
    .reset_index(drop=True)
)

# Identify observations that represent normal domestic venue demand.
venue_match_history["venue_attendance_history_eligible"] = (
    venue_match_history["is_domestic_match"]
    & ~venue_match_history["is_pandemic_affected_season"]
    & venue_match_history["has_positive_attendance"]
)

# Preserve the original attendance while masking observations that
# should not contribute to normal venue-attendance history.
venue_match_history["venue_attendance_for_history"] = (
    venue_match_history["attendance"].where(
        venue_match_history[
            "venue_attendance_history_eligible"
        ]
    )
)

# Shift before rolling so that the current match cannot use
# its own observed attendance.
venue_match_history["venue_last_10_attendance_mean"] = (
    venue_match_history
    .groupby("venue_name", sort=False)[
        "venue_attendance_for_history"
    ]
    .transform(
        lambda attendance:
        attendance
        .shift(1)
        .rolling(
            window=10,
            min_periods=1,
        )
        .mean()
    )
)

# Return the venue feature to the one-row-per-match working table.
venue_attendance_features = venue_match_history[
    [
        "match_id",
        "venue_last_10_attendance_mean",
    ]
]

historical_feature_working = historical_feature_working.merge(
    venue_attendance_features,
    on="match_id",
    how="left",
    validate="one_to_one",
)

eligible_venue_history_rows = int(
    venue_match_history[
        "venue_attendance_history_eligible"
    ].sum()
)

excluded_venue_history_rows = int(
    (
        ~venue_match_history[
            "venue_attendance_history_eligible"
        ]
    ).sum()
)

print(
    "Eligible venue-attendance history rows: "
    f"{eligible_venue_history_rows:,}"
)
print(
    "Venue-attendance history rows excluded from rolling calculations: "
    f"{excluded_venue_history_rows:,}"
)

print("MCG venue-history example:")

display(
    venue_match_history.loc[
        venue_match_history["venue_name"].eq("MCG"),
        [
            "match_id",
            "match_date",
            "venue_name",
            "attendance",
            "venue_attendance_for_history",
            "venue_last_10_attendance_mean",
        ],
    ].head(8)
)

display(
    historical_feature_working[
        [
            "match_id",
            "match_date",
            "home_team",
            "away_team",
            "venue_name",
            "attendance",
            "venue_last_10_attendance_mean",
        ]
    ].head(5)
)

print("Time-safe venue attendance feature creation: PASSED")

Eligible venue-attendance history rows: 2,504
Venue-attendance history rows excluded from rolling calculations: 375
MCG venue-history example:


,match_id,match_date,venue_name,attendance,venue_attendance_for_history,venue_last_10_attendance_mean
1485,20120329_H14_A03,2012-03-29,MCG,78285,78285.0,NaN
1486,20120330_H10_A04,2012-03-30,MCG,78466,78466.0,78285.000000
1487,20120331_H11_A02,2012-03-31,MCG,33473,33473.0,78375.500000
1488,20120407_H04_A14,2012-04-07,MCG,57268,57268.0,63408.000000
1489,20120409_H07_A10,2012-04-09,MCG,69231,69231.0,61873.000000
1490,20120413_H03_A04,2012-04-13,MCG,84259,84259.0,63344.600000
1491,20120414_H14_A11,2012-04-14,MCG,49826,49826.0,66830.333333
1492,20120415_H10_A01,2012-04-15,MCG,33524,33524.0,64401.142857


,match_id,match_date,home_team,away_team,venue_name,attendance,venue_last_10_attendance_mean
0,20120324_H09_A16,2012-03-24,Greater Western Sydney,Sydney,Stadium Australia,38203,NaN
1,20120329_H14_A03,2012-03-29,Richmond,Carlton,MCG,78285,NaN
2,20120330_H10_A04,2012-03-30,Hawthorn,Collingwood,MCG,78466,78285.0
3,20120331_H11_A02,2012-03-31,Melbourne,Brisbane Lions,MCG,33473,78375.5
4,20120331_H08_A01,2012-03-31,Gold Coast,Adelaide,Carrara,12790,NaN


Time-safe venue attendance feature creation: PASSED


### 4.4 Create Pre-Match Team Form Features

Recent team performance may influence supporter interest and expected match attendance.

Two features are calculated for each team:

- `team_last_5_win_rate`: the proportion of the previous five matches won by the team;
- `team_last_5_score_margin_mean`: the mean score margin from the previous five matches.

A positive score margin indicates that the team won by that number of points, while a negative value indicates a loss. A draw is not counted as a win when calculating the win rate.

The five-match window provides a simple measure of recent form while reducing the influence of one unusual result.

Unlike attendance-history features, match results from 2020 and 2021 are retained. These were publicly observed results available to supporters before later fixtures. This does not assume that team performance was unaffected by the pandemic, and the unusual match conditions remain a project limitation.

Both features are shifted by one match before the rolling calculation. Therefore, the current match result and score margin cannot contribute to their own predictor values.

In [10]:
# Calculate the proportion of the previous five matches won by each team.
team_match_history["team_last_5_win_rate"] = (
    team_match_history
    .groupby("team_id", sort=False)["team_won"]
    .transform(
        lambda results:
        results
        .astype(float)
        .shift(1)
        .rolling(
            window=5,
            min_periods=1,
        )
        .mean()
    )
)

# Calculate the mean score margin from the previous five matches.
team_match_history["team_last_5_score_margin_mean"] = (
    team_match_history
    .groupby("team_id", sort=False)["score_margin"]
    .transform(
        lambda margins:
        margins
        .shift(1)
        .rolling(
            window=5,
            min_periods=1,
        )
        .mean()
    )
)

# Return the temporary team-form features to one row per match.
team_form_features = team_match_history.pivot(
    index="match_id",
    columns="team_role",
    values=[
        "team_last_5_win_rate",
        "team_last_5_score_margin_mean",
    ],
)

team_form_features.columns = [
    f"{team_role}_{feature_name}"
    for feature_name, team_role
    in team_form_features.columns
]

team_form_features = team_form_features.reset_index()

historical_feature_working = historical_feature_working.merge(
    team_form_features,
    on="match_id",
    how="left",
    validate="one_to_one",
)

display(
    team_match_history[
        [
            "match_id",
            "match_date",
            "team_name",
            "team_role",
            "team_score",
            "opponent_score",
            "score_margin",
            "team_won",
            "team_last_5_win_rate",
            "team_last_5_score_margin_mean",
        ]
    ].head(8)
)

display(
    historical_feature_working[
        [
            "match_id",
            "match_date",
            "home_team",
            "away_team",
            "home_team_last_5_win_rate",
            "away_team_last_5_win_rate",
            "home_team_last_5_score_margin_mean",
            "away_team_last_5_score_margin_mean",
        ]
    ].head(5)
)

print("Time-safe pre-match team-form feature creation: PASSED")

,match_id,match_date,team_name,team_role,team_score,opponent_score,score_margin,team_won,team_last_5_win_rate,team_last_5_score_margin_mean
0,20120331_H08_A01,2012-03-31,Adelaide,away,137,68,69,True,NaN,NaN
1,20120407_H01_A18,2012-04-07,Adelaide,home,82,64,18,True,1.000000,69.000000
2,20120415_H10_A01,2012-04-15,Adelaide,away,84,140,-56,False,1.000000,43.500000
3,20120421_H01_A09,2012-04-21,Adelaide,home,96,50,46,True,0.666667,10.333333
4,20120429_H01_A13,2012-04-29,Adelaide,home,110,91,19,True,0.750000,19.250000
5,20120505_H16_A01,2012-05-05,Adelaide,away,99,94,5,True,0.800000,19.200000
6,20120512_H01_A07,2012-05-12,Adelaide,home,122,72,50,True,0.800000,6.400000
7,20120520_H03_A01,2012-05-20,Adelaide,away,124,55,69,True,0.800000,12.800000


,match_id,match_date,home_team,away_team,home_team_last_5_win_rate,away_team_last_5_win_rate,home_team_last_5_score_margin_mean,away_team_last_5_score_margin_mean
0,20120324_H09_A16,2012-03-24,Greater Western Sydney,Sydney,NaN,NaN,NaN,NaN
1,20120329_H14_A03,2012-03-29,Richmond,Carlton,NaN,NaN,NaN,NaN
2,20120330_H10_A04,2012-03-30,Hawthorn,Collingwood,NaN,NaN,NaN,NaN
3,20120331_H11_A02,2012-03-31,Melbourne,Brisbane Lions,NaN,NaN,NaN,NaN
4,20120331_H08_A01,2012-03-31,Gold Coast,Adelaide,NaN,NaN,NaN,NaN


Time-safe pre-match team-form feature creation: PASSED


### 4.5 Review Early-History Missing Values

Rolling historical features are unavailable when a team or venue does not have sufficient earlier match history.

These missing values are expected rather than being data-quality errors. They may occur:

- during the earliest matches in the dataset;
- when a team first appears in the historical data;
- when a venue hosts its first recorded match;
- after the pandemic period because 2020 and 2021 attendance is excluded from normal attendance-history calculations.

No historical match is removed solely because a rolling predictor is missing.

The 2012 season provides an initial history period before the main training years. Any remaining numerical missing values will be handled in the model-development notebook by an imputer fitted on the training data only.

Missing-value indicators may also be added by the preprocessing pipeline so that the model can distinguish an imputed value from an originally observed historical value.

In [11]:
history_feature_columns = [
    "home_team_last_5_attendance_mean",
    "away_team_last_5_attendance_mean",
    "venue_last_10_attendance_mean",
    "home_team_last_5_win_rate",
    "away_team_last_5_win_rate",
    "home_team_last_5_score_margin_mean",
    "away_team_last_5_score_margin_mean",
]

# Summarise missing values for the newly created historical features.
history_feature_missing_summary = (
    historical_feature_working[history_feature_columns]
    .isna()
    .sum()
    .rename("missing_rows")
    .to_frame()
)

history_feature_missing_summary["missing_percentage"] = (
    history_feature_missing_summary["missing_rows"]
    .div(len(historical_feature_working))
    .mul(100)
    .round(2)
)

display(history_feature_missing_summary)

# Show how rows with incomplete history are distributed by season.
rows_with_missing_history = (
    historical_feature_working[history_feature_columns]
    .isna()
    .any(axis=1)
)

missing_history_by_season = (
    historical_feature_working[
        ["season_year"]
    ]
    .assign(
        has_missing_history_feature=rows_with_missing_history
    )
    .groupby("season_year")[
        "has_missing_history_feature"
    ]
    .agg(
        total_rows="size",
        rows_with_missing="sum",
    )
    .reset_index()
)

missing_history_by_season["missing_percentage"] = (
    missing_history_by_season["rows_with_missing"]
    .div(missing_history_by_season["total_rows"])
    .mul(100)
    .round(2)
)

display(missing_history_by_season)

print(
    "Historical rows with at least one missing history feature: "
    f"{rows_with_missing_history.sum():,}"
)
print("Rows removed because of missing history features: 0")
print("Early-history missing-value review: COMPLETED")

,missing_rows,missing_percentage
home_team_last_5_attendance_mean,341,11.84
away_team_last_5_attendance_mean,343,11.91
venue_last_10_attendance_mean,292,10.14
home_team_last_5_win_rate,9,0.31
away_team_last_5_win_rate,9,0.31
home_team_last_5_score_margin_mean,9,0.31
away_team_last_5_score_margin_mean,9,0.31


,season_year,total_rows,rows_with_missing,missing_percentage
0,2012,207,19,9.18
1,2013,207,1,0.48
2,2014,207,3,1.45
3,2015,206,1,0.49
4,2016,207,0,0.00
5,2017,207,2,0.97
6,2018,207,2,0.97
7,2019,207,2,0.97
8,2020,162,119,73.46
9,2021,207,207,100.00


Historical rows with at least one missing history feature: 373
Rows removed because of missing history features: 0
Early-history missing-value review: COMPLETED


## 5. Define the Model-Ready Historical Dataset

### 5.1 Select the Target and Feature Columns

The historical model dataset contains four types of columns:

- one numerical prediction target: `attendance`;
- candidate feature columns that were available before each match;
- identifier and date columns retained for traceability and time-based splitting;
- temporary eligibility flags used to assign dataset roles.

The candidate features include scheduled match context, calendar information, school-holiday status, recent team attendance, recent venue attendance, and recent team form.

Current-match final scores are not selected. They were used only to create shifted historical team-form features and must not enter the model directly.

Identifier fields and eligibility flags are retained in the dataset but will not be treated as model features.

Categorical encoding, numerical imputation, feature scaling, and model fitting are not performed in this notebook. These steps will be fitted on the training data in the model-development pipeline.

In [12]:
target_column = "attendance"

# Select the target, candidate features, identifiers,
# and temporary dataset-control fields.
historical_model_candidate_columns = [
    # Record identity and time
    "match_id",
    "source_game_id",
    "season_year",
    "match_date",
    "match_datetime_local",
    "start_time",

    # Scheduled match and venue context
    "round_label",
    "home_team_id",
    "home_team",
    "away_team_id",
    "away_team",
    "venue_name",
    "venue_city",
    "state_code",
    "country_code",
    "school_holiday_city",

    # Numerical prediction target
    "attendance",

    # Calendar and scheduled context features
    "match_month",
    "match_day_of_week",
    "is_weekend",
    "start_hour",
    "is_night_match",
    "is_finals_match",
    "is_school_holiday",

    # Time-safe attendance-history features
    "home_team_last_5_attendance_mean",
    "away_team_last_5_attendance_mean",
    "venue_last_10_attendance_mean",

    # Time-safe team-form features
    "home_team_last_5_win_rate",
    "away_team_last_5_win_rate",
    "home_team_last_5_score_margin_mean",
    "away_team_last_5_score_margin_mean",

    # Temporary dataset-eligibility flags
    "is_domestic_match",
    "is_pandemic_affected_season",
    "has_positive_attendance",
]

historical_model_candidates = historical_feature_working[
    historical_model_candidate_columns
].copy()

print(
    f"Historical model candidate rows: "
    f"{len(historical_model_candidates):,}"
)
print(
    f"Selected columns: "
    f"{historical_model_candidates.shape[1]}"
)
print(f"Prediction target: {target_column}")

print("Recent historical model candidate sample:")

display(
    historical_model_candidates[
        [
            "match_id",
            "season_year",
            "match_date",
            "home_team",
            "away_team",
            "venue_name",
            "attendance",
            "home_team_last_5_attendance_mean",
            "away_team_last_5_attendance_mean",
            "venue_last_10_attendance_mean",
            "home_team_last_5_win_rate",
            "away_team_last_5_win_rate",
        ]
    ]
    .sort_values(
        ["match_date", "match_id"]
    )
    .tail(5)
)

print("Historical target and feature selection: COMPLETED")

Historical model candidate rows: 2,879
Selected columns: 34
Prediction target: attendance
Recent historical model candidate sample:


,match_id,season_year,match_date,home_team,away_team,venue_name,attendance,home_team_last_5_attendance_mean,away_team_last_5_attendance_mean,venue_last_10_attendance_mean,home_team_last_5_win_rate,away_team_last_5_win_rate
2874,20250912_H01_A10,2025,2025-09-12,Adelaide,Hawthorn,Adelaide Oval,52005,41492.6,45152.0,42291.8,0.8,0.6
2875,20250913_H02_A08,2025,2025-09-13,Brisbane Lions,Gold Coast,Gabba,36628,57300.4,31059.8,30657.1,0.6,0.6
2876,20250919_H07_A10,2025,2025-09-19,Geelong,Hawthorn,MCG,99567,45696.4,45422.2,57452.8,1.0,0.8
2877,20250920_H04_A02,2025,2025-09-20,Collingwood,Brisbane Lions,MCG,96023,63584.4,48160.8,61464.1,0.4,0.6
2878,20250927_H07_A02,2025,2025-09-27,Geelong,Brisbane Lions,MCG,100022,59997.0,61080.6,62833.8,1.0,0.8


Historical target and feature selection: COMPLETED


### 5.2 Assign Time-Based Dataset Roles

A random train-test split is not used because it could allow later matches to influence the evaluation of earlier periods.

Each historical row is assigned a chronological dataset role:

- `warm_up`: 2012, used to provide earlier history for the first training season;
- `training`: 2013–2019 and 2022–2023;
- `excluded`: 2020–2021 pandemic-affected matches and international matches;
- `validation`: 2024, used for model comparison and development decisions;
- `locked_test`: 2025, used once for final out-of-time evaluation.

The 2020 and 2021 seasons are excluded as target rows because attendance was directly affected by closures and capacity restrictions. Their earlier match results have already been retained where required for shifted team-form calculations.

International matches are excluded because the current forecasting scope covers domestic AFL attendance and Australian school-holiday information.

The 2012 warm-up rows are not passed to model fitting. Their purpose is to provide earlier history for matches from 2013 onward.

After final evaluation has been reported, the selected pipeline may be refitted using all eligible training, validation, and test-period rows before producing the 2026 forecasts.

In [13]:
# A valid model target must represent positive domestic attendance
# outside the pandemic-affected seasons.
eligible_target_mask = (
    historical_model_candidates["is_domestic_match"]
    & ~historical_model_candidates[
        "is_pandemic_affected_season"
    ]
    & historical_model_candidates[
        "has_positive_attendance"
    ]
)

historical_model_candidates["dataset_role"] = pd.Series(
    "unassigned",
    index=historical_model_candidates.index,
    dtype="string",
)

# Assign records that are outside the domestic target scope.
historical_model_candidates.loc[
    ~eligible_target_mask,
    "dataset_role",
] = "excluded"

# Use 2012 only to build history for later matches.
historical_model_candidates.loc[
    eligible_target_mask
    & historical_model_candidates["season_year"].eq(2012),
    "dataset_role",
] = "warm_up"

# Use normal pre-pandemic years and the first two post-pandemic
# seasons for model training.
training_year_mask = (
    historical_model_candidates["season_year"]
    .between(2013, 2019)
    | historical_model_candidates["season_year"]
    .between(2022, 2023)
)

historical_model_candidates.loc[
    eligible_target_mask & training_year_mask,
    "dataset_role",
] = "training"

# Reserve the two latest completed seasons for chronological evaluation.
historical_model_candidates.loc[
    eligible_target_mask
    & historical_model_candidates["season_year"].eq(2024),
    "dataset_role",
] = "validation"

historical_model_candidates.loc[
    eligible_target_mask
    & historical_model_candidates["season_year"].eq(2025),
    "dataset_role",
] = "locked_test"

unassigned_role_count = int(
    historical_model_candidates["dataset_role"]
    .eq("unassigned")
    .sum()
)

if unassigned_role_count > 0:
    raise ValueError(
        f"Historical rows without a dataset role: "
        f"{unassigned_role_count}"
    )

dataset_role_summary = (
    historical_model_candidates["dataset_role"]
    .value_counts()
    .rename_axis("dataset_role")
    .reset_index(name="records")
)

display(dataset_role_summary)

# Retain only rows that can be used for model development or evaluation.
model_dataset_roles = [
    "training",
    "validation",
    "locked_test",
]

historical_model_dataset = (
    historical_model_candidates.loc[
        historical_model_candidates[
            "dataset_role"
        ].isin(model_dataset_roles)
    ]
    .drop(
        columns=[
            "is_domestic_match",
            "is_pandemic_affected_season",
            "has_positive_attendance",
        ]
    )
    .sort_values(["match_date", "match_id"])
    .reset_index(drop=True)
)

print(
    f"Historical candidate rows: "
    f"{len(historical_model_candidates):,}"
)
print(
    f"Model-development rows retained: "
    f"{len(historical_model_dataset):,}"
)
print(
    "Time-based dataset-role assignment: PASSED"
)

,dataset_role,records
0,training,1865
1,excluded,375
2,validation,216
3,locked_test,216
4,warm_up,207


Historical candidate rows: 2,879
Model-development rows retained: 2,297
Time-based dataset-role assignment: PASSED


### 5.3 Define Numerical, Categorical, and Binary Features

The candidate features are divided into numerical, categorical, and binary feature groups. These groups will be passed to separate preprocessing steps in the model-development pipeline.

Numerical features contain ordered measurements such as start hour, recent attendance averages, win rates, and score margins.

Categorical features represent named groups such as teams, venues, rounds, months, and days of the week. `match_month` is treated as categorical because the difference between consecutive month numbers should not be interpreted as a fixed linear effect.

Binary features represent pre-match conditions with `True` or `False` values.

Several retained fields are intentionally excluded from the predictor lists:

- match and source identifiers are retained only for traceability;
- dates and raw start times are retained for chronological splitting and reporting;
- team IDs are identifiers rather than numerical measurements;
- venue location fields are retained for reporting but are already represented by `venue_name`;
- `is_weekend` is excluded because weekend information is already represented by `match_day_of_week`;
- `is_finals_match` is excluded because finals information is already represented by `round_label`;
- `dataset_role` controls the time split and is not a predictor;
- `attendance` is the prediction target.

No encoding, imputation, or scaling is fitted in this notebook.

In [14]:
# Define continuous or ordered numerical predictors.
# The historical averages are retained as numerical values and will
# be imputed within the training pipeline when required.
numerical_feature_columns = [
    "season_year",
    "start_hour",
    "home_team_last_5_attendance_mean",
    "away_team_last_5_attendance_mean",
    "venue_last_10_attendance_mean",
    "home_team_last_5_win_rate",
    "away_team_last_5_win_rate",
    "home_team_last_5_score_margin_mean",
    "away_team_last_5_score_margin_mean",
]

# Define named categories that require categorical encoding.
# Match month is treated as categorical to avoid assuming that its
# relationship with attendance is linear.
categorical_feature_columns = [
    "round_label",
    "home_team",
    "away_team",
    "venue_name",
    "match_month",
    "match_day_of_week",
]

# Define pre-match True-or-False conditions.
# Weekend and finals flags are not selected because their information
# is already represented by day-of-week and round-label features.
binary_feature_columns = [
    "is_night_match",
    "is_school_holiday",
]

# Combine the three feature groups into the complete model input list.
feature_columns = (
    numerical_feature_columns
    + categorical_feature_columns
    + binary_feature_columns
)

# Confirm that every selected predictor exists in the historical dataset.
missing_defined_features = sorted(
    set(feature_columns)
    - set(historical_model_dataset.columns)
)

if missing_defined_features:
    raise ValueError(
        "Defined features missing from the historical dataset: "
        + ", ".join(missing_defined_features)
    )

# Protect against accidentally including the observed attendance target
# in the predictor matrix.
if target_column in feature_columns:
    raise ValueError(
        "The prediction target must not appear in the feature list."
    )

# Create a compact feature register for notebook review and handoff.
feature_definition = pd.DataFrame(
    {
        "feature_name": feature_columns,
        "feature_type": (
            ["numerical"] * len(numerical_feature_columns)
            + ["categorical"] * len(categorical_feature_columns)
            + ["binary"] * len(binary_feature_columns)
        ),
    }
)

display(feature_definition)

# Report the final feature-group sizes without fitting any preprocessing.
print(
    f"Numerical features: "
    f"{len(numerical_feature_columns)}"
)
print(
    f"Categorical features: "
    f"{len(categorical_feature_columns)}"
)
print(
    f"Binary features: "
    f"{len(binary_feature_columns)}"
)
print(f"Total model features: {len(feature_columns)}")
print(f"Prediction target: {target_column}")
print("Historical model feature definition: COMPLETED")

,feature_name,feature_type
0,season_year,numerical
1,start_hour,numerical
2,home_team_last_5_attendance_mean,numerical
3,away_team_last_5_attendance_mean,numerical
4,venue_last_10_attendance_mean,numerical
5,home_team_last_5_win_rate,numerical
6,away_team_last_5_win_rate,numerical
7,home_team_last_5_score_margin_mean,numerical
8,away_team_last_5_score_margin_mean,numerical
9,round_label,categorical


Numerical features: 9
Categorical features: 6
Binary features: 2
Total model features: 17
Prediction target: attendance
Historical model feature definition: COMPLETED


### 5.4 Validate Missing Values and Data Leakage

This subsection performs a focused validation of the historical model dataset before the 2026 scoring dataset is prepared.

The checks confirm that:

- each retained match has a unique identifier;
- the attendance target is complete and positive;
- current-match outcomes are not included as predictors;
- missing values occur only in the time-safe historical features;
- the training, validation, and locked-test periods remain chronologically ordered.

Missing historical features are retained at this stage. Their treatment will be fitted using the training data only in the modelling notebook.

In [15]:
# Count missing values for each selected model feature.
feature_missing_summary = (
    historical_model_dataset[feature_columns]
    .isna()
    .sum()
    .rename("missing_rows")
    .to_frame()
)

feature_missing_summary["missing_percentage"] = (
    feature_missing_summary["missing_rows"]
    .div(len(historical_model_dataset))
    .mul(100)
    .round(2)
)

# Retain only features that contain at least one missing value.
features_with_missing_values = feature_missing_summary.loc[
    feature_missing_summary["missing_rows"] > 0
].copy()

# Historical rolling features may be missing when insufficient prior history exists.
non_history_feature_columns = [
    column
    for column in feature_columns
    if column not in history_feature_columns
]

unexpected_missing_columns = [
    column
    for column in non_history_feature_columns
    if historical_model_dataset[column].isna().any()
]

# Confirm that identifiers, targets, outcomes, and split labels are not predictors.
forbidden_feature_columns = {
    "match_id",
    "match_date",
    "match_datetime_local",
    "attendance",
    "home_score",
    "away_score",
    "has_positive_attendance",
    "is_pandemic_affected_season",
    "dataset_role",
}

leakage_feature_columns = sorted(
    set(feature_columns).intersection(forbidden_feature_columns)
)

# Summarise the chronological coverage of each model-development role.
dataset_role_order = ["training", "validation", "locked_test"]

dataset_role_date_summary = (
    historical_model_dataset
    .groupby("dataset_role", observed=True)
    .agg(
        records=("match_id", "size"),
        start_date=("match_date", "min"),
        end_date=("match_date", "max"),
    )
    .reindex(dataset_role_order)
    .reset_index()
)

role_dates = dataset_role_date_summary.set_index("dataset_role")

chronological_role_order = bool(
    role_dates.loc["training", "end_date"]
    < role_dates.loc["validation", "start_date"]
    < role_dates.loc["locked_test", "start_date"]
)

# Calculate the final validation measures.
duplicate_match_id_rows = int(
    historical_model_dataset["match_id"].duplicated().sum()
)

missing_target_rows = int(
    historical_model_dataset[target_column].isna().sum()
)

non_positive_target_rows = int(
    historical_model_dataset[target_column].le(0).sum()
)

rows_with_missing_history = int(
    historical_model_dataset[history_feature_columns]
    .isna()
    .any(axis=1)
    .sum()
)

validation_5_4 = pd.DataFrame(
    [
        {
            "check": "Duplicate match_id rows",
            "value": duplicate_match_id_rows,
            "expected": 0,
        },
        {
            "check": "Missing attendance targets",
            "value": missing_target_rows,
            "expected": 0,
        },
        {
            "check": "Non-positive attendance targets",
            "value": non_positive_target_rows,
            "expected": 0,
        },
        {
            "check": "Outcome or identifier columns used as features",
            "value": len(leakage_feature_columns),
            "expected": 0,
        },
        {
            "check": "Unexpected missing feature columns",
            "value": len(unexpected_missing_columns),
            "expected": 0,
        },
        {
            "check": "Chronological dataset-role order",
            "value": chronological_role_order,
            "expected": True,
        },
    ]
)

validation_5_4["status"] = [
    "PASSED" if value == expected else "FAILED"
    for value, expected in zip(
        validation_5_4["value"],
        validation_5_4["expected"],
    )
]

display(dataset_role_date_summary)

print("Selected features with missing values:")
display(features_with_missing_values)

display(validation_5_4)

# Stop execution if a required model-dataset check fails.
if validation_5_4["status"].eq("FAILED").any():
    raise ValueError(
        "The historical model dataset failed one or more validation checks."
    )

print(
    "Historical rows with at least one missing history feature: "
    f"{rows_with_missing_history:,}"
)
print("Rows removed because of missing history features: 0")
print("Historical model-dataset validation: PASSED")

,dataset_role,records,start_date,end_date
0,training,1865,2013-03-22,2023-09-30
1,validation,216,2024-03-07,2024-09-28
2,locked_test,216,2025-03-07,2025-09-27


Selected features with missing values:


,missing_rows,missing_percentage
home_team_last_5_attendance_mean,9,0.39
away_team_last_5_attendance_mean,9,0.39
venue_last_10_attendance_mean,18,0.78


,check,value,expected,status
0,Duplicate match_id rows,0,0,PASSED
1,Missing attendance targets,0,0,PASSED
2,Non-positive attendance targets,0,0,PASSED
3,Outcome or identifier columns used as features,0,0,PASSED
4,Unexpected missing feature columns,0,0,PASSED
5,Chronological dataset-role order,True,True,PASSED


Historical rows with at least one missing history feature: 22
Rows removed because of missing history features: 0
Historical model-dataset validation: PASSED


## 6. Prepare the 2026 Scoring Dataset

### 6.1 Select Fixtures Available at the Snapshot Date

The 2026 Squiggle snapshot contains completed matches, confirmed future fixtures, and finals placeholders. Only confirmed future fixtures are required for attendance scoring.

A selected scoring fixture must:

- occur after the API snapshot date;
- have confirmed home and away teams;
- have a confirmed date, start time, and venue;
- have unique source and match identifiers;
- contain no completed-match scores.

Completed matches and finals placeholders are excluded because they do not represent confirmed future matches requiring an attendance prediction.

In [16]:
# Confirm that the source contains one consistent snapshot date.
snapshot_date_values = (
    pd.to_datetime(
        squiggle_matches_2026["snapshot_date"],
        errors="coerce",
    )
    .dropna()
    .drop_duplicates()
)

if len(snapshot_date_values) != 1:
    raise ValueError(
        "The Squiggle source must contain exactly one snapshot date."
    )

forecast_snapshot_date = snapshot_date_values.iloc[0]

# Select confirmed future fixtures without modifying the source DataFrame.
scoring_fixtures_2026 = (
    squiggle_matches_2026.loc[
        squiggle_matches_2026["snapshot_record_type"].eq(
            "future_fixture"
        )
    ]
    .copy()
)

if scoring_fixtures_2026.empty:
    raise ValueError(
        "No confirmed future fixtures were found in the snapshot."
    )

# Standardise the date columns used in the scoring workflow.
scoring_fixtures_2026["snapshot_date"] = pd.to_datetime(
    scoring_fixtures_2026["snapshot_date"],
    errors="coerce",
)

scoring_fixtures_2026["match_date"] = pd.to_datetime(
    scoring_fixtures_2026["match_date"],
    errors="coerce",
)

# Define the fields required to identify and score each fixture.
required_scoring_fields = [
    "snapshot_date",
    "source_game_id",
    "match_id",
    "match_date",
    "start_time",
    "round_label",
    "home_team",
    "away_team",
    "venue_name",
]

missing_required_values = int(
    scoring_fixtures_2026[required_scoring_fields]
    .isna()
    .sum()
    .sum()
)

duplicate_source_game_ids = int(
    scoring_fixtures_2026["source_game_id"]
    .duplicated()
    .sum()
)

duplicate_match_ids = int(
    scoring_fixtures_2026["match_id"]
    .duplicated()
    .sum()
)

fixtures_not_after_snapshot = int(
    scoring_fixtures_2026["match_date"]
    .le(forecast_snapshot_date)
    .sum()
)

# Future fixtures must not contain completed-match scores.
fixtures_with_scores = int(
    scoring_fixtures_2026[
        ["home_score", "away_score"]
    ]
    .notna()
    .any(axis=1)
    .sum()
)

fixture_selection_validation = pd.DataFrame(
    {
        "check": [
            "Missing required fixture values",
            "Duplicate source_game_id rows",
            "Duplicate match_id rows",
            "Fixtures on or before snapshot date",
            "Future fixtures containing scores",
        ],
        "value": [
            missing_required_values,
            duplicate_source_game_ids,
            duplicate_match_ids,
            fixtures_not_after_snapshot,
            fixtures_with_scores,
        ],
        "expected": [0, 0, 0, 0, 0],
    }
)

fixture_selection_validation["status"] = [
    "PASSED" if value == expected else "FAILED"
    for value, expected in zip(
        fixture_selection_validation["value"],
        fixture_selection_validation["expected"],
    )
]

# Present the selected fixtures in chronological order.
scoring_fixtures_2026 = (
    scoring_fixtures_2026
    .sort_values(
        ["match_date", "start_time", "source_game_id"]
    )
    .reset_index(drop=True)
)

fixture_preview_columns = [
    "snapshot_date",
    "source_game_id",
    "match_id",
    "match_date",
    "start_time",
    "round_label",
    "home_team",
    "away_team",
    "venue_name",
]

print(f"Forecast snapshot date: {forecast_snapshot_date.date()}")
print(f"Future fixtures selected: {len(scoring_fixtures_2026):,}")

display(
    scoring_fixtures_2026[fixture_preview_columns]
)

display(fixture_selection_validation)

# Stop execution if any required fixture check fails.
if fixture_selection_validation["status"].eq("FAILED").any():
    raise ValueError(
        "The 2026 scoring-fixture selection failed validation."
    )

print("2026 scoring-fixture selection check: PASSED")

Forecast snapshot date: 2026-08-17
Future fixtures selected: 9


,snapshot_date,source_game_id,match_id,match_date,start_time,round_label,home_team,away_team,venue_name
0,2026-08-17,38697,20260820_H15_A08,2026-08-20,19:30:00,Round 24,St Kilda,Gold Coast,Docklands
1,2026-08-17,38692,20260821_H04_A02,2026-08-21,19:40:00,Round 24,Collingwood,Brisbane Lions,MCG
2,2026-08-17,38693,20260822_H03_A06,2026-08-22,13:15:00,Round 24,Carlton,Fremantle,Docklands
3,2026-08-17,38696,20260822_H11_A18,2026-08-22,16:15:00,Round 24,Melbourne,Western Bulldogs,MCG
4,2026-08-17,38699,20260822_H01_A09,2026-08-22,19:40:00,Round 24,Adelaide,Greater Western Sydney,Adelaide Oval
5,2026-08-17,38695,20260822_H07_A14,2026-08-22,19:45:00,Round 24,Geelong,Richmond,Kardinia Park
6,2026-08-17,38694,20260823_H05_A13,2026-08-23,12:20:00,Round 24,Essendon,Port Adelaide,Docklands
7,2026-08-17,38698,20260823_H16_A12,2026-08-23,15:20:00,Round 24,Sydney,North Melbourne,SCG
8,2026-08-17,38700,20260823_H17_A10,2026-08-23,17:20:00,Round 24,West Coast,Hawthorn,Perth Stadium


,check,value,expected,status
0,Missing required fixture values,0,0,PASSED
1,Duplicate source_game_id rows,0,0,PASSED
2,Duplicate match_id rows,0,0,PASSED
3,Fixtures on or before snapshot date,0,0,PASSED
4,Future fixtures containing scores,0,0,PASSED


2026 scoring-fixture selection check: PASSED


### 6.2 Apply the Match-Context Transformations

The selected 2026 fixtures are transformed using the same calendar and match-context rules applied to the historical dataset.

The following pre-match features are created:

- calendar month;
- day of the week;
- weekend status;
- numerical start hour;
- night-match status;
- finals status;
- school-holiday status for the venue's associated capital city.

Venue location fields were already standardised during data preparation. The school-holiday status is joined using the venue's mapped holiday city and the fixture date.

All information used in this subsection was available on the 17 August 2026 snapshot date. Historical attendance and team-form features are added separately in the next subsection.

In [17]:
# Preserve the selected scoring-fixture table before adding context features.
scoring_context_2026 = scoring_fixtures_2026.copy()
original_scoring_row_count = len(scoring_context_2026)

# Standardise the fixture date before deriving calendar features.
scoring_context_2026["match_date"] = pd.to_datetime(
    scoring_context_2026["match_date"],
    errors="coerce",
)

# Create calendar features using the same definitions as the historical dataset.
scoring_context_2026["season_year"] = (
    scoring_context_2026["match_date"]
    .dt.year
    .astype("Int64")
)

scoring_context_2026["match_month"] = (
    scoring_context_2026["match_date"]
    .dt.month
    .astype("Int64")
)

scoring_context_2026["match_day_of_week"] = (
    scoring_context_2026["match_date"]
    .dt.day_name()
    .astype("string")
)

scoring_context_2026["is_weekend"] = (
    scoring_context_2026["match_day_of_week"]
    .isin(["Saturday", "Sunday"])
)

# Convert the scheduled start time into a numerical hour.
parsed_start_times = pd.to_datetime(
    scoring_context_2026["start_time"].astype("string"),
    format="%H:%M:%S",
    errors="coerce",
)

scoring_context_2026["start_hour"] = (
    parsed_start_times.dt.hour
    + parsed_start_times.dt.minute.div(60)
)

# Apply the same night-match threshold used for historical matches.
scoring_context_2026["is_night_match"] = (
    scoring_context_2026["start_hour"].ge(17)
)

# Derive finals status consistently from the round label.
scoring_context_2026["is_finals_match"] = (
    scoring_context_2026["round_label"]
    .astype("string")
    .str.contains("Final", case=False, na=False)
)

scoring_context_2026["is_domestic_match"] = (
    scoring_context_2026["country_code"].eq("AU")
)

# Prepare the city-date holiday lookup for the scoring fixtures.
scoring_holiday_lookup = (
    school_holidays_daily[
        ["city", "calendar_date", "is_school_holiday"]
    ]
    .rename(
        columns={
            "city": "school_holiday_city",
            "calendar_date": "match_date",
        }
    )
    .copy()
)

scoring_holiday_lookup["match_date"] = pd.to_datetime(
    scoring_holiday_lookup["match_date"],
    errors="coerce",
)

# Attach the school-holiday flag using venue city and fixture date.
scoring_context_2026 = scoring_context_2026.merge(
    scoring_holiday_lookup,
    on=["school_holiday_city", "match_date"],
    how="left",
    validate="many_to_one",
)

scoring_context_2026["is_school_holiday"] = (
    scoring_context_2026["is_school_holiday"]
    .astype("boolean")
)

# Check that the fixture rows and required context values remain complete.
venue_context_columns = [
    "venue_city",
    "country_code",
    "school_holiday_city",
]

created_context_columns = [
    "season_year",
    "match_month",
    "match_day_of_week",
    "start_hour",
    "is_night_match",
    "is_school_holiday",
]

missing_venue_context_rows = int(
    scoring_context_2026[venue_context_columns]
    .isna()
    .any(axis=1)
    .sum()
)

missing_domestic_holiday_rows = int(
    (
        scoring_context_2026["is_domestic_match"]
        & scoring_context_2026["is_school_holiday"].isna()
    ).sum()
)

missing_created_context_values = int(
    scoring_context_2026[created_context_columns]
    .isna()
    .sum()
    .sum()
)

context_validation_6_2 = pd.DataFrame(
    {
        "check": [
            "Fixture rows preserved",
            "Missing venue-context rows",
            "Missing domestic holiday rows",
            "Missing created context values",
        ],
        "value": [
            len(scoring_context_2026),
            missing_venue_context_rows,
            missing_domestic_holiday_rows,
            missing_created_context_values,
        ],
        "expected": [
            original_scoring_row_count,
            0,
            0,
            0,
        ],
    }
)

context_validation_6_2["status"] = [
    "PASSED" if value == expected else "FAILED"
    for value, expected in zip(
        context_validation_6_2["value"],
        context_validation_6_2["expected"],
    )
]

context_preview_columns = [
    "match_id",
    "match_date",
    "start_time",
    "home_team",
    "away_team",
    "venue_name",
    "venue_city",
    "match_month",
    "match_day_of_week",
    "start_hour",
    "is_night_match",
    "is_school_holiday",
]

display(
    scoring_context_2026[context_preview_columns]
)

display(context_validation_6_2)

# Stop execution if the scoring context is incomplete.
if context_validation_6_2["status"].eq("FAILED").any():
    raise ValueError(
        "The 2026 match-context transformation failed validation."
    )

print("2026 match-context feature creation: PASSED")

,match_id,match_date,start_time,home_team,away_team,venue_name,venue_city,match_month,match_day_of_week,start_hour,is_night_match,is_school_holiday
0,20260820_H15_A08,2026-08-20,19:30:00,St Kilda,Gold Coast,Docklands,Melbourne,8,Thursday,19.500000,True,False
1,20260821_H04_A02,2026-08-21,19:40:00,Collingwood,Brisbane Lions,MCG,Melbourne,8,Friday,19.666667,True,False
2,20260822_H03_A06,2026-08-22,13:15:00,Carlton,Fremantle,Docklands,Melbourne,8,Saturday,13.250000,False,False
3,20260822_H11_A18,2026-08-22,16:15:00,Melbourne,Western Bulldogs,MCG,Melbourne,8,Saturday,16.250000,False,False
4,20260822_H01_A09,2026-08-22,19:40:00,Adelaide,Greater Western Sydney,Adelaide Oval,Adelaide,8,Saturday,19.666667,True,False
5,20260822_H07_A14,2026-08-22,19:45:00,Geelong,Richmond,Kardinia Park,Geelong,8,Saturday,19.750000,True,False
6,20260823_H05_A13,2026-08-23,12:20:00,Essendon,Port Adelaide,Docklands,Melbourne,8,Sunday,12.333333,False,False
7,20260823_H16_A12,2026-08-23,15:20:00,Sydney,North Melbourne,SCG,Sydney,8,Sunday,15.333333,False,False
8,20260823_H17_A10,2026-08-23,17:20:00,West Coast,Hawthorn,Perth Stadium,Perth,8,Sunday,17.333333,True,False


,check,value,expected,status
0,Fixture rows preserved,9,9,PASSED
1,Missing venue-context rows,0,0,PASSED
2,Missing domestic holiday rows,0,0,PASSED
3,Missing created context values,0,0,PASSED


2026 match-context feature creation: PASSED


### 6.3 Attach the Latest Available Historical Features

The scoring fixtures require the same seven time-safe historical features used in the historical model dataset.

The attendance-based features use the latest eligible attendance records available from the historical match source:

- each team's mean attendance across its latest five eligible matches;
- each venue's mean attendance across its latest ten eligible matches.

Pandemic-affected seasons, zero-attendance matches, and international matches remain excluded from these attendance calculations.

The Squiggle snapshot does not provide 2026 attendance figures. Therefore, the attendance-based features use the latest reliable attendance information available through the end of the 2025 historical source.

**The latest Audience History**

In [18]:
# Select matches eligible for attendance-history calculations.
attendance_history_source = (
    historical_model_candidates.loc[
        ~historical_model_candidates[
            "is_pandemic_affected_season"
        ]
        & historical_model_candidates["is_domestic_match"]
        & historical_model_candidates["has_positive_attendance"]
    ]
    .copy()
)

# Reshape home and away teams into one team-match history table.
home_team_attendance = (
    attendance_history_source[
        ["match_id", "match_date", "home_team", "attendance"]
    ]
    .rename(columns={"home_team": "team_name"})
)

away_team_attendance = (
    attendance_history_source[
        ["match_id", "match_date", "away_team", "attendance"]
    ]
    .rename(columns={"away_team": "team_name"})
)

team_attendance_history_for_scoring = pd.concat(
    [home_team_attendance, away_team_attendance],
    ignore_index=True,
)

team_attendance_history_for_scoring = (
    team_attendance_history_for_scoring
    .sort_values(["team_name", "match_date", "match_id"])
)

# Calculate each team's mean attendance over its latest five eligible matches.
latest_team_attendance = (
    team_attendance_history_for_scoring
    .groupby("team_name", group_keys=False)
    .tail(5)
    .groupby("team_name", as_index=False)
    .agg(
        team_last_5_attendance_mean=(
            "attendance",
            "mean",
        )
    )
)

# Calculate each venue's mean attendance over its latest ten eligible matches.
latest_venue_attendance = (
    attendance_history_source
    .sort_values(["venue_name", "match_date", "match_id"])
    .groupby("venue_name", group_keys=False)
    .tail(10)
    .groupby("venue_name", as_index=False)
    .agg(
        venue_last_10_attendance_mean=(
            "attendance",
            "mean",
        )
    )
)

print(
    "Eligible attendance-history matches: "
    f"{len(attendance_history_source):,}"
)
print(
    "Latest attendance date available: "
    f"{attendance_history_source['match_date'].max().date()}"
)
print(
    "Team attendance lookups created: "
    f"{len(latest_team_attendance):,}"
)
print(
    "Venue attendance lookups created: "
    f"{len(latest_venue_attendance):,}"
)

Eligible attendance-history matches: 2,504
Latest attendance date available: 2025-09-27
Team attendance lookups created: 18
Venue attendance lookups created: 25


The team-form features can be updated further than the attendance-based features because the Squiggle snapshot contains the scores of completed 2026 matches.

Historical results through 2025 are combined with the 2026 matches completed on or before the snapshot date. Each team's latest five completed matches are then used to calculate:

- recent win rate;
- recent mean score margin.

No shift is required at this point because every result used in the calculation occurred before the future scoring fixtures.

**The updated Team Situation**

In [19]:
# Reuse the historical team-level result table created in Section 4.1.
historical_team_results_for_scoring = (
    team_match_history[
        [
            "match_id",
            "match_date",
            "team_name",
            "team_score",
            "opponent_score",
        ]
    ]
    .copy()
)

historical_team_results_for_scoring["match_date"] = pd.to_datetime(
    historical_team_results_for_scoring["match_date"],
    errors="coerce",
)

# Recalculate the historical result fields for a consistent structure.
historical_team_results_for_scoring["score_margin"] = (
    historical_team_results_for_scoring["team_score"]
    - historical_team_results_for_scoring["opponent_score"]
)

historical_team_results_for_scoring["team_won"] = (
    historical_team_results_for_scoring["score_margin"].gt(0)
)

# Select 2026 matches completed on or before the API snapshot date.
completed_2026_results = (
    squiggle_matches_2026.loc[
        squiggle_matches_2026["snapshot_record_type"].eq(
            "completed_match"
        ),
        [
            "match_id",
            "match_date",
            "home_team",
            "away_team",
            "home_score",
            "away_score",
        ],
    ]
    .copy()
)

completed_2026_results["match_date"] = pd.to_datetime(
    completed_2026_results["match_date"],
    errors="coerce",
)

completed_2026_results = (
    completed_2026_results.loc[
        completed_2026_results["match_date"].le(
            forecast_snapshot_date
        )
    ]
    .dropna(
        subset=[
            "home_team",
            "away_team",
            "home_score",
            "away_score",
        ]
    )
)

# Reshape the completed 2026 results into one row per team and match.
completed_2026_home_results = (
    completed_2026_results[
        [
            "match_id",
            "match_date",
            "home_team",
            "home_score",
            "away_score",
        ]
    ]
    .rename(
        columns={
            "home_team": "team_name",
            "home_score": "team_score",
            "away_score": "opponent_score",
        }
    )
)

completed_2026_away_results = (
    completed_2026_results[
        [
            "match_id",
            "match_date",
            "away_team",
            "away_score",
            "home_score",
        ]
    ]
    .rename(
        columns={
            "away_team": "team_name",
            "away_score": "team_score",
            "home_score": "opponent_score",
        }
    )
)

completed_2026_team_results = pd.concat(
    [
        completed_2026_home_results,
        completed_2026_away_results,
    ],
    ignore_index=True,
)

# Calculate each team's result for the completed 2026 matches.
completed_2026_team_results["score_margin"] = (
    completed_2026_team_results["team_score"]
    - completed_2026_team_results["opponent_score"]
)

completed_2026_team_results["team_won"] = (
    completed_2026_team_results["score_margin"].gt(0)
)

# Combine historical results with all 2026 results available at the snapshot date.
combined_results_for_scoring = pd.concat(
    [
        historical_team_results_for_scoring,
        completed_2026_team_results,
    ],
    ignore_index=True,
)

team_results_for_scoring = (
    combined_results_for_scoring
    .sort_values(["team_name", "match_date", "match_id"])
)

# Summarise each team's latest five completed matches.
latest_team_form = (
    team_results_for_scoring
    .groupby("team_name", group_keys=False)
    .tail(5)
    .groupby("team_name", as_index=False)
    .agg(
        team_last_5_win_rate=("team_won", "mean"),
        team_last_5_score_margin_mean=(
            "score_margin",
            "mean",
        ),
    )
)

print(
    "Historical team-result rows included: "
    f"{len(historical_team_results_for_scoring):,}"
)
print(
    "Completed 2026 matches included: "
    f"{len(completed_2026_results):,}"
)
print(
    "Completed 2026 team-result rows included: "
    f"{len(completed_2026_team_results):,}"
)
print(
    "Latest team-form result date: "
    f"{team_results_for_scoring['match_date'].max().date()}"
)
print(
    "Team-form lookups created: "
    f"{len(latest_team_form):,}"
)

Historical team-result rows included: 5,758
Completed 2026 matches included: 198
Completed 2026 team-result rows included: 396
Latest team-form result date: 2026-08-16
Team-form lookups created: 18


The latest attendance and team-form values are now mapped to the home team, away team, and venue of each future fixture.

This produces the same seven historical feature columns used by the historical model dataset. The fixture rows remain unchanged, and no future match result or attendance value is used.

**Consecutive Seven Historical Features**

In [20]:
# Preserve the context-enriched scoring fixtures.
scoring_features_2026 = scoring_context_2026.copy()
scoring_row_count_before_history = len(scoring_features_2026)

# Convert the latest historical summaries into mapping Series.
team_attendance_lookup = (
    latest_team_attendance
    .set_index("team_name")["team_last_5_attendance_mean"]
)

venue_attendance_lookup = (
    latest_venue_attendance
    .set_index("venue_name")["venue_last_10_attendance_mean"]
)

team_win_rate_lookup = (
    latest_team_form
    .set_index("team_name")["team_last_5_win_rate"]
)

team_score_margin_lookup = (
    latest_team_form
    .set_index("team_name")[
        "team_last_5_score_margin_mean"
    ]
)

# Attach the latest team attendance and form features.
for team_role in ["home", "away"]:
    team_column = f"{team_role}_team"

    scoring_features_2026[
        f"{team_role}_team_last_5_attendance_mean"
    ] = scoring_features_2026[team_column].map(
        team_attendance_lookup
    )

    scoring_features_2026[
        f"{team_role}_team_last_5_win_rate"
    ] = scoring_features_2026[team_column].map(
        team_win_rate_lookup
    )

    scoring_features_2026[
        f"{team_role}_team_last_5_score_margin_mean"
    ] = scoring_features_2026[team_column].map(
        team_score_margin_lookup
    )

# Attach the latest venue attendance-history feature.
scoring_features_2026[
    "venue_last_10_attendance_mean"
] = scoring_features_2026["venue_name"].map(
    venue_attendance_lookup
)

# Summarise missing values across the seven historical features.
scoring_history_missing_summary = (
    scoring_features_2026[history_feature_columns]
    .isna()
    .sum()
    .rename("missing_rows")
    .to_frame()
)

# Confirm that no match result after the snapshot date was used.
results_after_snapshot = int(
    combined_results_for_scoring["match_date"]
    .gt(forecast_snapshot_date)
    .sum()
)

missing_scoring_history_values = int(
    scoring_features_2026[history_feature_columns]
    .isna()
    .sum()
    .sum()
)

history_attachment_validation = pd.DataFrame(
    {
        "check": [
            "Fixture rows preserved",
            "Result rows after snapshot date",
            "Missing scoring-history values",
        ],
        "value": [
            len(scoring_features_2026),
            results_after_snapshot,
            missing_scoring_history_values,
        ],
        "expected": [
            scoring_row_count_before_history,
            0,
            0,
        ],
    }
)

history_attachment_validation["status"] = [
    "PASSED" if value == expected else "FAILED"
    for value, expected in zip(
        history_attachment_validation["value"],
        history_attachment_validation["expected"],
    )
]

# Define separate previews to keep the notebook output readable.
attendance_feature_preview = [
    "match_id",
    "home_team",
    "away_team",
    "venue_name",
    "home_team_last_5_attendance_mean",
    "away_team_last_5_attendance_mean",
    "venue_last_10_attendance_mean",
]

form_feature_preview = [
    "match_id",
    "home_team",
    "away_team",
    "home_team_last_5_win_rate",
    "away_team_last_5_win_rate",
    "home_team_last_5_score_margin_mean",
    "away_team_last_5_score_margin_mean",
]

print("2026 scoring attendance-history features:")

display(
    scoring_features_2026[attendance_feature_preview]
)

print("2026 scoring team-form features:")

display(
    scoring_features_2026[form_feature_preview]
)

display(scoring_history_missing_summary)
display(history_attachment_validation)

# Stop execution if the attached history is incomplete or time-unsafe.
if history_attachment_validation["status"].eq("FAILED").any():
    raise ValueError(
        "The 2026 historical-feature attachment failed validation."
    )

print("2026 historical-feature attachment: PASSED")

2026 scoring attendance-history features:


,match_id,home_team,away_team,venue_name,home_team_last_5_attendance_mean,away_team_last_5_attendance_mean,venue_last_10_attendance_mean
0,20260820_H15_A08,St Kilda,Gold Coast,Docklands,27522.6,33561.4,25785.7
1,20260821_H04_A02,Collingwood,Brisbane Lions,MCG,66323.8,70224.6,65984.5
2,20260822_H03_A06,Carlton,Fremantle,Docklands,36183.8,45125.2,25785.7
3,20260822_H11_A18,Melbourne,Western Bulldogs,MCG,37635.8,30654.8,65984.5
4,20260822_H01_A09,Adelaide,Greater Western Sydney,Adelaide Oval,41762.8,16571.0,44454.2
5,20260822_H07_A14,Geelong,Richmond,Kardinia Park,73943.6,34595.0,30108.4
6,20260823_H05_A13,Essendon,Port Adelaide,Docklands,28023.8,34077.8,25785.7
7,20260823_H16_A12,Sydney,North Melbourne,SCG,28890.0,19569.2,33673.6
8,20260823_H17_A10,West Coast,Hawthorn,Perth Stadium,32171.8,51632.6,44788.9


2026 scoring team-form features:


,match_id,home_team,away_team,home_team_last_5_win_rate,away_team_last_5_win_rate,home_team_last_5_score_margin_mean,away_team_last_5_score_margin_mean
0,20260820_H15_A08,St Kilda,Gold Coast,0.4,0.2,-2.0,-14.2
1,20260821_H04_A02,Collingwood,Brisbane Lions,0.6,0.8,9.8,23.2
2,20260822_H03_A06,Carlton,Fremantle,0.6,0.8,22.4,31.0
3,20260822_H11_A18,Melbourne,Western Bulldogs,0.8,0.4,17.0,1.2
4,20260822_H01_A09,Adelaide,Greater Western Sydney,0.6,0.4,7.8,2.0
5,20260822_H07_A14,Geelong,Richmond,1.0,0.2,30.8,-34.8
6,20260823_H05_A13,Essendon,Port Adelaide,0.2,0.0,-53.0,-54.4
7,20260823_H16_A12,Sydney,North Melbourne,0.8,0.2,37.4,-11.0
8,20260823_H17_A10,West Coast,Hawthorn,0.0,0.6,-35.6,22.4


,missing_rows
home_team_last_5_attendance_mean,0
away_team_last_5_attendance_mean,0
venue_last_10_attendance_mean,0
home_team_last_5_win_rate,0
away_team_last_5_win_rate,0
home_team_last_5_score_margin_mean,0
away_team_last_5_score_margin_mean,0


,check,value,expected,status
0,Fixture rows preserved,9,9,PASSED
1,Result rows after snapshot date,0,0,PASSED
2,Missing scoring-history values,0,0,PASSED


2026 historical-feature attachment: PASSED


### 6.4 Align the Historical and Scoring Schemas

The historical and 2026 scoring datasets must present the same model features in the same order and with compatible data types.

The historical dataset contains:

- match identifiers and dates;
- the dataset role used for chronological model evaluation;
- the attendance target;
- the 17 selected model features.

The 2026 scoring dataset contains:

- snapshot and match identifiers;
- fixture dates and times;
- the same 17 model features;
- no attendance target.

Only deterministic data-type alignment is performed here. Missing-value imputation, categorical encoding, scaling, and model fitting remain part of the modelling workflow.

In [21]:
# Define the non-feature columns retained for identification and evaluation.
historical_metadata_columns = [
    "source_game_id",
    "match_id",
    "match_date",
    "start_time",
    "dataset_role",
]

scoring_metadata_columns = [
    "snapshot_date",
    "source_game_id",
    "match_id",
    "match_date",
    "start_time",
]

# Build the historical output with its target and selected model features.
historical_model_ready = (
    historical_model_dataset[
        historical_metadata_columns
        + [target_column]
        + feature_columns
    ]
    .copy()
)

# Build the future scoring output without an attendance target.
scoring_model_ready_2026 = (
    scoring_features_2026[
        scoring_metadata_columns
        + feature_columns
    ]
    .copy()
)

# Standardise the metadata date columns.
historical_model_ready["match_date"] = pd.to_datetime(
    historical_model_ready["match_date"],
    errors="coerce",
)

scoring_model_ready_2026["snapshot_date"] = pd.to_datetime(
    scoring_model_ready_2026["snapshot_date"],
    errors="coerce",
)

scoring_model_ready_2026["match_date"] = pd.to_datetime(
    scoring_model_ready_2026["match_date"],
    errors="coerce",
)

# Keep season year as an integer-valued numerical feature.
for dataset in [
    historical_model_ready,
    scoring_model_ready_2026,
]:
    dataset["season_year"] = pd.to_numeric(
        dataset["season_year"],
        errors="coerce",
    ).astype("Int64")

# Standardise the remaining continuous numerical features.
continuous_numerical_features = [
    column
    for column in numerical_feature_columns
    if column != "season_year"
]

for dataset in [
    historical_model_ready,
    scoring_model_ready_2026,
]:
    for column in continuous_numerical_features:
        dataset[column] = pd.to_numeric(
            dataset[column],
            errors="coerce",
        ).astype("Float64")

    # Represent categorical features consistently as strings.
    for column in categorical_feature_columns:
        dataset[column] = dataset[column].astype("string")

    # Represent binary features consistently as nullable booleans.
    for column in binary_feature_columns:
        dataset[column] = dataset[column].astype("boolean")

# Present both output datasets in chronological order.
historical_model_ready = (
    historical_model_ready
    .sort_values(["match_date", "match_id"])
    .reset_index(drop=True)
)

scoring_model_ready_2026 = (
    scoring_model_ready_2026
    .sort_values(["match_date", "start_time", "match_id"])
    .reset_index(drop=True)
)

The aligned schemas are checked before export.

The validation confirms that:

- both datasets contain the same 17 model features in the same order;
- each corresponding feature has the same data type;
- the attendance target appears only in the historical dataset;
- all nine future fixtures are preserved;
- the future scoring features are complete.

Expected early-history missing values remain in the historical dataset and will be handled by the training-only preprocessing pipeline.

In [22]:
# Compare the selected feature order across the two output datasets.
historical_feature_order = (
    historical_model_ready[feature_columns]
    .columns
    .tolist()
)

scoring_feature_order = (
    scoring_model_ready_2026[feature_columns]
    .columns
    .tolist()
)

feature_order_matches = (
    historical_feature_order == scoring_feature_order
)

# Record the intended modelling role of every selected feature.
feature_type_lookup = {
    **{
        column: "numerical"
        for column in numerical_feature_columns
    },
    **{
        column: "categorical"
        for column in categorical_feature_columns
    },
    **{
        column: "binary"
        for column in binary_feature_columns
    },
}

# Compare the aligned pandas data types feature by feature.
feature_schema_comparison = pd.DataFrame(
    {
        "feature_name": feature_columns,
        "feature_type": [
            feature_type_lookup[column]
            for column in feature_columns
        ],
        "historical_dtype": [
            str(historical_model_ready[column].dtype)
            for column in feature_columns
        ],
        "scoring_dtype": [
            str(scoring_model_ready_2026[column].dtype)
            for column in feature_columns
        ],
    }
)

feature_schema_comparison["dtype_match"] = (
    feature_schema_comparison["historical_dtype"]
    .eq(feature_schema_comparison["scoring_dtype"])
)

all_feature_dtypes_match = bool(
    feature_schema_comparison["dtype_match"].all()
)

missing_scoring_feature_values = int(
    scoring_model_ready_2026[feature_columns]
    .isna()
    .sum()
    .sum()
)

historical_rows_with_missing_features = int(
    historical_model_ready[feature_columns]
    .isna()
    .any(axis=1)
    .sum()
)

schema_validation_6_4 = pd.DataFrame(
    {
        "check": [
            "Model feature order matches",
            "Model feature data types match",
            "Historical attendance target present",
            "Attendance target excluded from scoring",
            "Scoring fixture rows preserved",
            "Missing scoring feature values",
        ],
        "value": [
            feature_order_matches,
            all_feature_dtypes_match,
            target_column in historical_model_ready.columns,
            target_column not in scoring_model_ready_2026.columns,
            len(scoring_model_ready_2026),
            missing_scoring_feature_values,
        ],
        "expected": [
            True,
            True,
            True,
            True,
            len(scoring_features_2026),
            0,
        ],
    }
)

schema_validation_6_4["status"] = [
    "PASSED" if value == expected else "FAILED"
    for value, expected in zip(
        schema_validation_6_4["value"],
        schema_validation_6_4["expected"],
    )
]

display(feature_schema_comparison)
display(schema_validation_6_4)

print(
    "Historical model-ready dataset: "
    f"{len(historical_model_ready):,} rows × "
    f"{historical_model_ready.shape[1]} columns"
)

print(
    "2026 scoring dataset: "
    f"{len(scoring_model_ready_2026):,} rows × "
    f"{scoring_model_ready_2026.shape[1]} columns"
)

print(
    "Historical rows with at least one missing feature: "
    f"{historical_rows_with_missing_features:,}"
)

# Stop execution if the historical and scoring schemas are incompatible.
if schema_validation_6_4["status"].eq("FAILED").any():
    raise ValueError(
        "The historical and scoring schemas are not aligned."
    )

print("Historical and scoring schema alignment: PASSED")

,feature_name,feature_type,historical_dtype,scoring_dtype,dtype_match
0,season_year,numerical,Int64,Int64,True
1,start_hour,numerical,Float64,Float64,True
2,home_team_last_5_attendance_mean,numerical,Float64,Float64,True
3,away_team_last_5_attendance_mean,numerical,Float64,Float64,True
4,venue_last_10_attendance_mean,numerical,Float64,Float64,True
5,home_team_last_5_win_rate,numerical,Float64,Float64,True
6,away_team_last_5_win_rate,numerical,Float64,Float64,True
7,home_team_last_5_score_margin_mean,numerical,Float64,Float64,True
8,away_team_last_5_score_margin_mean,numerical,Float64,Float64,True
9,round_label,categorical,string,string,True


,check,value,expected,status
0,Model feature order matches,True,True,PASSED
1,Model feature data types match,True,True,PASSED
2,Historical attendance target present,True,True,PASSED
3,Attendance target excluded from scoring,True,True,PASSED
4,Scoring fixture rows preserved,9,9,PASSED
5,Missing scoring feature values,0,0,PASSED


Historical model-ready dataset: 2,297 rows × 23 columns
2026 scoring dataset: 9 rows × 22 columns
Historical rows with at least one missing feature: 22
Historical and scoring schema alignment: PASSED


## 7. Validate, Export, and Handoff

### 7.1 Final Dataset Validation and Summary

This subsection provides a short pre-export checkpoint for the two final datasets. Detailed transformation checks were already completed in Sections 5 and 6 and are not repeated here.

The final check confirms that:

- all prepared rows are preserved;
- match identifiers remain unique;
- both datasets contain the same 17 ordered model features;
- the attendance target appears only in the historical dataset;
- the 2026 scoring features contain no missing values.

Expected missing rolling-history values are retained in the historical dataset for training-only preprocessing in the modelling notebook.

In [23]:
# Confirm that both outputs contain the same ordered model features.
feature_schema_matches = (
    historical_model_ready[feature_columns]
    .columns
    .tolist()
    == scoring_model_ready_2026[feature_columns]
    .columns
    .tolist()
)

# Calculate the small set of measures required before export.
historical_duplicate_ids = int(
    historical_model_ready["match_id"]
    .duplicated()
    .sum()
)

scoring_duplicate_ids = int(
    scoring_model_ready_2026["match_id"]
    .duplicated()
    .sum()
)

historical_missing_target_rows = int(
    historical_model_ready[target_column]
    .isna()
    .sum()
)

historical_missing_feature_rows = int(
    historical_model_ready[feature_columns]
    .isna()
    .any(axis=1)
    .sum()
)

scoring_missing_feature_rows = int(
    scoring_model_ready_2026[feature_columns]
    .isna()
    .any(axis=1)
    .sum()
)

# Apply concise readiness rules to the two final outputs.
historical_ready = bool(
    len(historical_model_ready)
    == len(historical_model_dataset)
    and historical_duplicate_ids == 0
    and historical_missing_target_rows == 0
    and target_column in historical_model_ready.columns
    and feature_schema_matches
)

scoring_ready = bool(
    len(scoring_model_ready_2026)
    == len(scoring_features_2026)
    and scoring_duplicate_ids == 0
    and scoring_missing_feature_rows == 0
    and target_column not in scoring_model_ready_2026.columns
    and feature_schema_matches
)

# Summarise the final deliverables in one compact table.
final_dataset_summary = pd.DataFrame(
    [
        {
            "dataset": "historical_model_ready",
            "records": len(historical_model_ready),
            "columns": historical_model_ready.shape[1],
            "model_features": len(feature_columns),
            "date_coverage": (
                f"{historical_model_ready['match_date'].min().date()} "
                f"to "
                f"{historical_model_ready['match_date'].max().date()}"
            ),
            "target": target_column,
            "duplicate_match_ids": historical_duplicate_ids,
            "missing_feature_rows": historical_missing_feature_rows,
            "status": "PASSED" if historical_ready else "FAILED",
        },
        {
            "dataset": "scoring_model_ready_2026",
            "records": len(scoring_model_ready_2026),
            "columns": scoring_model_ready_2026.shape[1],
            "model_features": len(feature_columns),
            "date_coverage": (
                f"{scoring_model_ready_2026['match_date'].min().date()} "
                f"to "
                f"{scoring_model_ready_2026['match_date'].max().date()}"
            ),
            "target": "Not applicable",
            "duplicate_match_ids": scoring_duplicate_ids,
            "missing_feature_rows": scoring_missing_feature_rows,
            "status": "PASSED" if scoring_ready else "FAILED",
        },
    ]
)

display(final_dataset_summary)

print(
    "Shared ordered feature schema: "
    f"{'PASSED' if feature_schema_matches else 'FAILED'}"
)

# Stop execution before export if either dataset is not ready.
if not historical_ready or not scoring_ready:
    raise ValueError(
        "One or more final datasets are not ready for export."
    )

print("Final model-ready dataset check: PASSED")

,dataset,records,columns,model_features,date_coverage,target,duplicate_match_ids,missing_feature_rows,status
0,historical_model_ready,2297,23,17,2013-03-22 to 2025-09-27,attendance,0,22,PASSED
1,scoring_model_ready_2026,9,22,17,2026-08-20 to 2026-08-23,Not applicable,0,0,PASSED


Shared ordered feature schema: PASSED
Final model-ready dataset check: PASSED


### 7.2 Export the Model-Ready Files

The validated historical and 2026 scoring datasets are exported as CSV files for the modelling workflow.

The outputs are:

- `historical_model_dataset.csv`, containing the historical target, dataset roles, and 17 model features;
- `squiggle_2026_scoring_dataset.csv`, containing the nine confirmed future fixtures and the same 17 model features without an attendance target.

The files are written to `data/processed/`. This step does not modify the raw data or write the feature tables back to PostgreSQL.

In [24]:
# Resolve the project root from either the project or notebooks directory.
export_project_root = Path.cwd().resolve()

if export_project_root.name == "notebooks":
    export_project_root = export_project_root.parent

# Create the processed-data directory if it does not already exist.
processed_output_dir = (
    export_project_root
    / "data"
    / "processed"
)

processed_output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

# Define the two model-ready output paths.
historical_output_path = (
    processed_output_dir
    / "historical_model_dataset.csv"
)

scoring_output_path = (
    processed_output_dir
    / "squiggle_2026_scoring_dataset.csv"
)

# Export the historical training and evaluation dataset.
historical_model_ready.to_csv(
    historical_output_path,
    index=False,
    date_format="%Y-%m-%d",
)

# Export the 2026 future-fixture scoring dataset.
scoring_model_ready_2026.to_csv(
    scoring_output_path,
    index=False,
    date_format="%Y-%m-%d",
)

# Summarise the two exported deliverables.
export_summary = pd.DataFrame(
    [
        {
            "file_name": historical_output_path.name,
            "record_grain": "One completed historical match",
            "records": len(historical_model_ready),
            "columns": historical_model_ready.shape[1],
            "target": target_column,
            "output_path": str(
                historical_output_path.relative_to(
                    export_project_root
                )
            ),
            "status": (
                "EXPORTED"
                if historical_output_path.exists()
                else "FAILED"
            ),
        },
        {
            "file_name": scoring_output_path.name,
            "record_grain": "One confirmed future fixture",
            "records": len(scoring_model_ready_2026),
            "columns": scoring_model_ready_2026.shape[1],
            "target": "Not applicable",
            "output_path": str(
                scoring_output_path.relative_to(
                    export_project_root
                )
            ),
            "status": (
                "EXPORTED"
                if scoring_output_path.exists()
                else "FAILED"
            ),
        },
    ]
)

display(export_summary)

# Stop execution if either output file was not created.
if export_summary["status"].eq("FAILED").any():
    raise OSError(
        "One or more model-ready files were not exported."
    )

print("Model-ready dataset export: PASSED")

,file_name,record_grain,records,columns,target,output_path,status
0,historical_model_dataset.csv,One completed historical match,2297,23,attendance,data\processed\historical_model_dataset.csv,EXPORTED
1,squiggle_2026_scoring_dataset.csv,One confirmed future fixture,9,22,Not applicable,data\processed\squiggle_2026_scoring_dataset.csv,EXPORTED


Model-ready dataset export: PASSED


### 7.3 Limitations and Next Step

#### 7.3.1 Current Limitations

The model-ready datasets are suitable for the v1 attendance-forecasting workflow, but several limitations should be considered when interpreting future model results:

- The project uses public match and calendar data rather than internal club ticketing, membership, or customer data.
- Reliable attendance records are available through the end of the 2025 historical source. The Squiggle snapshot provides 2026 match results but does not provide 2026 attendance figures.
- Team-form features use completed match results available through 16 August 2026, immediately before the 17 August 2026 snapshot date.
- Attendance-affected matches from the 2020 and 2021 pandemic seasons are excluded from attendance modelling and attendance-history calculations. Their observed match results remain available for calculating subsequent team form.
- The v1 feature set does not include ticket prices, membership status, marketing activity, confirmed player line-ups, injuries, or weather forecasts.
- The scoring dataset contains the nine confirmed Round 24 fixtures available in the frozen snapshot. Finals placeholders are excluded because the participating teams were not yet confirmed.
- International matches are excluded from the model-development dataset, so the resulting model is designed for Australian AFL fixtures.
- A small number of historical rows retain missing rolling-history features because insufficient eligible prior information was available. These values will be handled using training-only preprocessing.

### 7.3.2 Next Steps

Model development will continue across three focused notebooks.

#### `05_candidate_model_training_and_tuning.ipynb`

This notebook will:

1. load the historical model dataset;
2. isolate the training partition and separate the features from the attendance target;
3. define preprocessing pipelines for numerical and categorical features;
4. implement time-aware cross-validation within the training period;
5. tune the candidate regression model families;
6. retain the best configuration from each model family for subsequent comparison.

#### `06_model_comparison_and_evaluation.ipynb`

This notebook will:

1. establish a simple attendance baseline;
2. evaluate the tuned candidate models on the 2024 validation partition;
3. compare their forecasting performance using consistent evaluation metrics;
4. select and freeze the final model specification;
5. evaluate the selected model once on the locked 2025 test partition.

#### `07_attendance_forecasting_and_business_outputs.ipynb`

This notebook will:

1. refit the selected modelling workflow using all eligible historical data after final evaluation;
2. load the 2026 scoring dataset;
3. generate attendance forecasts for the nine confirmed 2026 fixtures;
4. assign demand tiers and operational review flags;
5. export the forecasting and model-performance outputs for downstream Power BI reporting and AWS deployment.

No model preprocessing, fitting, tuning, selection, evaluation, or prediction is performed in this notebook.